# 07. Day6 — 1단계·2단계 baseline (로지스틱 + RF)

- 입력: `data/processed/master_features.parquet` (05_seller_avg_delay.ipynb 산출물)
- 분할: 시간 기준(D-03) — 학습 `purchase_ts < 2018-05-01`, 테스트 `2018-05-01 ~ 2018-09-01`
- 1단계 입력: STAGE1(D-16), 2단계 입력: STAGE1+STAGE2, 기본 `class_weight='balanced'` (2단계 로지스틱만 None도 비교; 7절 CV 튜닝에서 2단계는 **미적용으로 확정**, D-20)
- 지표: PR-AUC(average_precision_score) 기본, ROC-AUC·혼동행렬(threshold=0.5)은 참고용
- 범주형(category_grouped)은 학습 데이터만으로 원-핫 인코딩을 fit하고 테스트에 동일하게 적용(인코딩 자체의 정보 누출 방지)


## 0. 데이터 로드 + 분할

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              confusion_matrix, classification_report)

master = pd.read_parquet("data/processed/master_features.parquet")
assert len(master) == 96204

STAGE1_NUM = ["distance_km_filled", "seller_avg_delay"]
STAGE1_BIN = ["dist_missing", "seller_avg_delay_is_fallback"]
STAGE1_CAT = ["category_grouped"]
STAGE2_NUM = ["price_sum", "freight_sum", "installments_max"]

TRAIN_END = "2018-05-01"   # D-03: 학습 < TRAIN_END, 테스트 = N=4(2018-05~08)
train_mask = master["purchase_ts"] < TRAIN_END
test_mask = (master["purchase_ts"] >= TRAIN_END) & (master["purchase_ts"] < "2018-09-01")

print(f"학습: {train_mask.sum()}건 / 테스트: {test_mask.sum()}건")
print(f"학습 late율: {master.loc[train_mask,'late'].mean():.4f} / 테스트 late율: {master.loc[test_mask,'late'].mean():.4f}")


학습: 70852건 / 테스트: 25352건
학습 late율: 0.0765 / 테스트 late율: 0.0440


## 1. 공통 유틸 — 전처리 파이프라인 + 평가 함수

`ColumnTransformer`를 학습 데이터에만 `fit`하고 테스트에는 `transform`만 적용한다(원-핫 인코딩 카테고리·스케일러 평균/표준편차 모두 학습 기준).

> **임계값 0.5 해석 주의**: `class_weight='balanced'`는 양성 클래스에 큰 가중치를 줘서 예측확률 전체를 끌어올린다.
> 그래서 임계값 0.5는 "확률 50%"의 의미가 아니고, 1단계 로지스틱처럼 전체의 약 60%를 양성으로 예측하기도 한다.
> 아래 혼동행렬은 참고용이며, 최종 보고용 혼동행렬은 TimeSeriesSplit 도입 후 CV로 임계값을 정해 다시 그린다.


In [2]:
def make_preprocessor(num_cols, bin_cols, cat_cols):
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("bin", "passthrough", bin_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])


def evaluate(name, y_true, y_prob):
    pr_auc = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)
    base_rate = y_true.mean()
    y_pred = (y_prob >= 0.5).astype(int)
    print(f"--- {name} ---")
    print(f"기저율(양성비율): {base_rate:.4f}")
    print(f"PR-AUC : {pr_auc:.4f}  (참고: 기저율만큼이 '무작위' 수준)")
    print(f"ROC-AUC: {roc_auc:.4f}")
    print("혼동행렬(threshold=0.5):")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=3))
    return {"name": name, "pr_auc": pr_auc, "roc_auc": roc_auc, "base_rate": base_rate}


results = []


## 2. 1단계 — 지연(late) 예측

입력: STAGE1(D-16). 모집단: 96,204건 전체(배송완료 주문).


In [3]:
X_cols_1 = STAGE1_NUM + STAGE1_BIN + STAGE1_CAT
X_train_1 = master.loc[train_mask, X_cols_1]
X_test_1 = master.loc[test_mask, X_cols_1]
y_train_1 = master.loc[train_mask, "late"]
y_test_1 = master.loc[test_mask, "late"]

pre_1 = make_preprocessor(STAGE1_NUM, STAGE1_BIN, STAGE1_CAT)

logit_1 = Pipeline([
    ("pre", pre_1),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
logit_1.fit(X_train_1, y_train_1)
prob_logit_1 = logit_1.predict_proba(X_test_1)[:, 1]
results.append(evaluate("1단계 로지스틱", y_test_1, prob_logit_1))


--- 1단계 로지스틱 ---
기저율(양성비율): 0.0440
PR-AUC : 0.0524  (참고: 기저율만큼이 '무작위' 수준)
ROC-AUC: 0.5062
혼동행렬(threshold=0.5):
[[ 9716 14521]
 [  460   655]]
              precision    recall  f1-score   support

           0      0.955     0.401     0.565     24237
           1      0.043     0.587     0.080      1115

    accuracy                          0.409     25352
   macro avg      0.499     0.494     0.323     25352
weighted avg      0.915     0.409     0.543     25352



In [4]:
rf_1 = Pipeline([
    ("pre", pre_1),
    ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                    max_depth=8, min_samples_leaf=50,
                                    random_state=42, n_jobs=-1)),
])
rf_1.fit(X_train_1, y_train_1)
prob_rf_1 = rf_1.predict_proba(X_test_1)[:, 1]
results.append(evaluate("1단계 RF", y_test_1, prob_rf_1))


--- 1단계 RF ---
기저율(양성비율): 0.0440
PR-AUC : 0.0519  (참고: 기저율만큼이 '무작위' 수준)
ROC-AUC: 0.4812
혼동행렬(threshold=0.5):
[[ 9529 14708]
 [  519   596]]
              precision    recall  f1-score   support

           0      0.948     0.393     0.556     24237
           1      0.039     0.535     0.073      1115

    accuracy                          0.399     25352
   macro avg      0.494     0.464     0.314     25352
weighted avg      0.908     0.399     0.535     25352



### 1단계 Feature Importance

**해석 주의 (먼저 읽을 것)**: 이 모델은 테스트에서 PR-AUC·ROC-AUC가 기저율/무작위 수준이므로,
아래 RF 중요도(MDI)는 **"지연에 기여하는 요인"의 근거로 쓰지 않는다.**
- MDI는 학습 데이터에서 트리가 변수를 얼마나 썼는가를 재는 값이라, 테스트 예측 기여와 다르다.
- 연속형 변수(`distance_km_filled`, `seller_avg_delay`)는 분할 후보가 많아 노이즈여도 값이 크게 나오는 편향이 있다.

참고로만 남기는 맥락: `seller_avg_delay`는 D-01 한계("fallback 행이 non-fallback보다 AUC가 높음 — 셀러 신호가 시기 효과를 대리")가 있다.
원인 진단은 6절(변수별 단독 AUC, 월별 추이)에서 한다.


In [5]:
feat_names = (STAGE1_NUM + STAGE1_BIN +
              list(rf_1.named_steps["pre"].named_transformers_["cat"].get_feature_names_out(STAGE1_CAT)))
importances = pd.Series(rf_1.named_steps["clf"].feature_importances_, index=feat_names)
print(importances.sort_values(ascending=False).head(10))


distance_km_filled                           0.457244
seller_avg_delay                             0.437222
seller_avg_delay_is_fallback                 0.029557
category_grouped_bed_bath_table              0.011570
category_grouped_cool_stuff                  0.006691
category_grouped_fashion_bags_accessories    0.006066
category_grouped_baby                        0.005266
category_grouped_luggage_accessories         0.004096
category_grouped_computers_accessories       0.003907
category_grouped_home_appliances             0.003803
dtype: float64


## 3. 2단계 — 저평점(low) 예측

입력: STAGE1 + STAGE2. 모집단: `low`가 NULL이 아닌 주문(review_id 있는 주문)만 — 1단계보다 표본이 작음(계획서 4절).

class_weight 처리: 이 분할에서 저평점 비율은 학습 13.8% / 테스트 10.0%로, 지연(학습 7.65% / 테스트 4.40%)보다 덜 불균형하다.
계획서에 따라 1단계와 별도로 판단하되, **로지스틱에서만 balanced/None을 비교**했고 RF는 balanced 하나만 실행했다.
순위 지표(PR-AUC·ROC-AUC)는 class_weight에 거의 영향받지 않았다(로지스틱 0.1362 vs 0.1355).
차이는 임계값 0.5의 혼동행렬에서만 크게 나타난다(None은 재현율 0.008).


In [6]:
stage2_mask_train = train_mask & master["low"].notna()
stage2_mask_test = test_mask & master["low"].notna()

X_cols_2 = STAGE1_NUM + STAGE2_NUM + STAGE1_BIN + STAGE1_CAT
X_train_2 = master.loc[stage2_mask_train, X_cols_2]
X_test_2 = master.loc[stage2_mask_test, X_cols_2]
y_train_2 = master.loc[stage2_mask_train, "low"].astype(int)
y_test_2 = master.loc[stage2_mask_test, "low"].astype(int)

print(f"2단계 학습: {len(X_train_2)}건 (양성비율 {y_train_2.mean():.4f})")
print(f"2단계 테스트: {len(X_test_2)}건 (양성비율 {y_test_2.mean():.4f})")

pre_2 = make_preprocessor(STAGE1_NUM + STAGE2_NUM, STAGE1_BIN, STAGE1_CAT)

for weight in ["balanced", None]:
    logit_2 = Pipeline([
        ("pre", pre_2),
        ("clf", LogisticRegression(class_weight=weight, max_iter=1000, random_state=42)),
    ])
    logit_2.fit(X_train_2, y_train_2)
    prob = logit_2.predict_proba(X_test_2)[:, 1]
    results.append(evaluate(f"2단계 로지스틱 (class_weight={weight})", y_test_2, prob))


2단계 학습: 70319건 (양성비율 0.1380)
2단계 테스트: 25242건 (양성비율 0.1000)
--- 2단계 로지스틱 (class_weight=balanced) ---
기저율(양성비율): 0.1000
PR-AUC : 0.1362  (참고: 기저율만큼이 '무작위' 수준)
ROC-AUC: 0.5755
혼동행렬(threshold=0.5):
[[10736 11981]
 [  948  1577]]
              precision    recall  f1-score   support

           0      0.919     0.473     0.624     22717
           1      0.116     0.625     0.196      2525

    accuracy                          0.488     25242
   macro avg      0.518     0.549     0.410     25242
weighted avg      0.839     0.488     0.581     25242

--- 2단계 로지스틱 (class_weight=None) ---
기저율(양성비율): 0.1000
PR-AUC : 0.1355  (참고: 기저율만큼이 '무작위' 수준)
ROC-AUC: 0.5733
혼동행렬(threshold=0.5):
[[22662    55]
 [ 2506    19]]
              precision    recall  f1-score   support

           0      0.900     0.998     0.947     22717
           1      0.257     0.008     0.015      2525

    accuracy                          0.899     25242
   macro avg      0.579     0.503     0.481     25242
weighted avg  

In [7]:
rf_2 = Pipeline([
    ("pre", pre_2),
    ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                    max_depth=8, min_samples_leaf=50,
                                    random_state=42, n_jobs=-1)),
])
rf_2.fit(X_train_2, y_train_2)
prob_rf_2 = rf_2.predict_proba(X_test_2)[:, 1]
results.append(evaluate("2단계 RF", y_test_2, prob_rf_2))


--- 2단계 RF ---
기저율(양성비율): 0.1000
PR-AUC : 0.1338  (참고: 기저율만큼이 '무작위' 수준)
ROC-AUC: 0.5706
혼동행렬(threshold=0.5):
[[10084 12633]
 [  942  1583]]
              precision    recall  f1-score   support

           0      0.915     0.444     0.598     22717
           1      0.111     0.627     0.189      2525

    accuracy                          0.462     25242
   macro avg      0.513     0.535     0.393     25242
weighted avg      0.834     0.462     0.557     25242



## 4. 결과 요약 + 1·2단계 변수중요도 대조

계획서 5절: 1단계 지연 연관 변수가 2단계에서도 상위인지 확인 — 겹치면 지연이 저평점의 공통 경로,
안 겹치면 서로 다른 메커니즘(3단계에서 다룰 "배송과 무관한 저평점"과 연결).
**주의**: 1·2단계는 모집단이 다름(2단계는 review_id 없는 주문 추가 제외로 표본이 더 작음) — 대조 결과
해석 시 이 표본 차이를 감안할 것.


In [8]:
summary = pd.DataFrame(results)[["name", "base_rate", "pr_auc", "roc_auc"]]
print(summary.to_string(index=False))

feat_names_2 = (STAGE1_NUM + STAGE2_NUM + STAGE1_BIN +
                 list(rf_2.named_steps["pre"].named_transformers_["cat"].get_feature_names_out(STAGE1_CAT)))
importances_2 = pd.Series(rf_2.named_steps["clf"].feature_importances_, index=feat_names_2)

common = importances.index.intersection(importances_2.index)
compare = pd.DataFrame({"1단계": importances[common], "2단계": importances_2[common]}).sort_values("1단계", ascending=False)
print("\n1·2단계 공통 변수 importance 대조:\n", compare)


                            name  base_rate   pr_auc  roc_auc
                        1단계 로지스틱   0.043981 0.052395 0.506158
                          1단계 RF   0.043981 0.051857 0.481212
2단계 로지스틱 (class_weight=balanced)   0.100032 0.136227 0.575453
    2단계 로지스틱 (class_weight=None)   0.100032 0.135534 0.573270
                          2단계 RF   0.100032 0.133767 0.570620

1·2단계 공통 변수 importance 대조:
                                                 1단계       2단계
distance_km_filled                         0.457244  0.156708
seller_avg_delay                           0.437222  0.200817
seller_avg_delay_is_fallback               0.029557  0.012466
category_grouped_bed_bath_table            0.011570  0.026558
category_grouped_cool_stuff                0.006691  0.005651
category_grouped_fashion_bags_accessories  0.006066  0.002745
category_grouped_baby                      0.005266  0.002993
category_grouped_luggage_accessories       0.004096  0.009595
category_grouped_computers_accessories   

## 5. [Ablation] purchase_month 포함 여부 — D-15 재검증

D-15에서 "연도별로 스파이크 시점이 다르다(2017-11 vs 2018-02~03)"는 그래프 관찰만으로 미사용을 확정했음.
하지만 계획서 5절의 TimeSeriesSplit/하이퍼파라미터 튜닝 같은 **데이터 기반 검증**을 거치지 않은 결정이었음.
1단계 모델 성능이 기저율 수준(PR-AUC 0.052)으로 거의 신호가 없었던 것과 맞물려,
"뺀 변수가 실은 거의 유일한 신호였을 가능성"을 직접 확인한다.

계획서 3절이 제시한 두 인코딩 방식을 그대로 각각 테스트:
- **(A) 정수 인코딩**: purchase_month를 1~12 그대로 숫자로 추가
- **(B) 성수기 더미**: 계획서 예시 그대로 11~12월=1, 나머지=0 (관찰된 스파이크에 맞춰 새로 정의하면
  결과를 보고 유리하게 고르는 것이 되므로, 계획서 원문의 정의를 그대로 사용)

비교 기준: 원래 STAGE1 모델(PR-AUC 0.0524/0.0519, ROC-AUC 0.5062/0.4812)과 PR-AUC·ROC-AUC 차이.

In [9]:
# (A) purchase_month 정수 인코딩 추가
STAGE1_NUM_A = STAGE1_NUM + ["purchase_month"]

X_train_1a = master.loc[train_mask, STAGE1_NUM_A + STAGE1_BIN + STAGE1_CAT]
X_test_1a  = master.loc[test_mask,  STAGE1_NUM_A + STAGE1_BIN + STAGE1_CAT]

pre_1a = make_preprocessor(STAGE1_NUM_A, STAGE1_BIN, STAGE1_CAT)

logit_1a = Pipeline([
    ("pre", pre_1a),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
logit_1a.fit(X_train_1a, y_train_1)
prob_logit_1a = logit_1a.predict_proba(X_test_1a)[:, 1]
results.append(evaluate("1단계 로지스틱 + purchase_month(정수)", y_test_1, prob_logit_1a))

rf_1a = Pipeline([
    ("pre", pre_1a),
    ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                    max_depth=8, min_samples_leaf=50,
                                    random_state=42, n_jobs=-1)),
])
rf_1a.fit(X_train_1a, y_train_1)
prob_rf_1a = rf_1a.predict_proba(X_test_1a)[:, 1]
results.append(evaluate("1단계 RF + purchase_month(정수)", y_test_1, prob_rf_1a))

--- 1단계 로지스틱 + purchase_month(정수) ---
기저율(양성비율): 0.0440
PR-AUC : 0.0526  (참고: 기저율만큼이 '무작위' 수준)
ROC-AUC: 0.5051
혼동행렬(threshold=0.5):
[[10290 13947]
 [  503   612]]
              precision    recall  f1-score   support

           0      0.953     0.425     0.587     24237
           1      0.042     0.549     0.078      1115

    accuracy                          0.430     25352
   macro avg      0.498     0.487     0.333     25352
weighted avg      0.913     0.430     0.565     25352

--- 1단계 RF + purchase_month(정수) ---
기저율(양성비율): 0.0440
PR-AUC : 0.0488  (참고: 기저율만큼이 '무작위' 수준)
ROC-AUC: 0.4829
혼동행렬(threshold=0.5):
[[22193  2044]
 [  970   145]]
              precision    recall  f1-score   support

           0      0.958     0.916     0.936     24237
           1      0.066     0.130     0.088      1115

    accuracy                          0.881     25352
   macro avg      0.512     0.523     0.512     25352
weighted avg      0.919     0.881     0.899     25352



In [10]:
# (B) purchase_month 성수기 더미 (계획서 예시 그대로: 11~12월=1)
master["peak_season"] = master["purchase_month"].isin([11, 12]).astype(int)
STAGE1_BIN_B = STAGE1_BIN + ["peak_season"]

X_train_1b = master.loc[train_mask, STAGE1_NUM + STAGE1_BIN_B + STAGE1_CAT]
X_test_1b  = master.loc[test_mask,  STAGE1_NUM + STAGE1_BIN_B + STAGE1_CAT]

pre_1b = make_preprocessor(STAGE1_NUM, STAGE1_BIN_B, STAGE1_CAT)

logit_1b = Pipeline([
    ("pre", pre_1b),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
logit_1b.fit(X_train_1b, y_train_1)
prob_logit_1b = logit_1b.predict_proba(X_test_1b)[:, 1]
results.append(evaluate("1단계 로지스틱 + peak_season(11~12월)", y_test_1, prob_logit_1b))

rf_1b = Pipeline([
    ("pre", pre_1b),
    ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                    max_depth=8, min_samples_leaf=50,
                                    random_state=42, n_jobs=-1)),
])
rf_1b.fit(X_train_1b, y_train_1)
prob_rf_1b = rf_1b.predict_proba(X_test_1b)[:, 1]
results.append(evaluate("1단계 RF + peak_season(11~12월)", y_test_1, prob_rf_1b))

--- 1단계 로지스틱 + peak_season(11~12월) ---
기저율(양성비율): 0.0440
PR-AUC : 0.0525  (참고: 기저율만큼이 '무작위' 수준)
ROC-AUC: 0.5072
혼동행렬(threshold=0.5):
[[11875 12362]
 [  561   554]]
              precision    recall  f1-score   support

           0      0.955     0.490     0.648     24237
           1      0.043     0.497     0.079      1115

    accuracy                          0.490     25352
   macro avg      0.499     0.493     0.363     25352
weighted avg      0.915     0.490     0.623     25352

--- 1단계 RF + peak_season(11~12월) ---
기저율(양성비율): 0.0440
PR-AUC : 0.0526  (참고: 기저율만큼이 '무작위' 수준)
ROC-AUC: 0.4820
혼동행렬(threshold=0.5):
[[ 9592 14645]
 [  523   592]]
              precision    recall  f1-score   support

           0      0.948     0.396     0.558     24237
           1      0.039     0.531     0.072      1115

    accuracy                          0.402     25352
   macro avg      0.494     0.463     0.315     25352
weighted avg      0.908     0.402     0.537     25352



In [11]:
# 비교 요약 — 원래 결과 대비 PR-AUC/ROC-AUC 변화
ablation_summary = pd.DataFrame(results)[["name", "base_rate", "pr_auc", "roc_auc"]]
print(ablation_summary.to_string(index=False))

base_pr  = ablation_summary.loc[ablation_summary.name == "1단계 로지스틱", "pr_auc"].iloc[0]
base_roc = ablation_summary.loc[ablation_summary.name == "1단계 로지스틱", "roc_auc"].iloc[0]
print(f"\n원래 1단계 로지스틱 대비 변화:")
for name in ["1단계 로지스틱 + purchase_month(정수)", "1단계 로지스틱 + peak_season(11~12월)"]:
    row = ablation_summary.loc[ablation_summary.name == name].iloc[0]
    print(f"  {name}: PR-AUC {row.pr_auc - base_pr:+.4f}, ROC-AUC {row.roc_auc - base_roc:+.4f}")

                            name  base_rate   pr_auc  roc_auc
                        1단계 로지스틱   0.043981 0.052395 0.506158
                          1단계 RF   0.043981 0.051857 0.481212
2단계 로지스틱 (class_weight=balanced)   0.100032 0.136227 0.575453
    2단계 로지스틱 (class_weight=None)   0.100032 0.135534 0.573270
                          2단계 RF   0.100032 0.133767 0.570620
   1단계 로지스틱 + purchase_month(정수)   0.043981 0.052614 0.505082
     1단계 RF + purchase_month(정수)   0.043981 0.048832 0.482910
  1단계 로지스틱 + peak_season(11~12월)   0.043981 0.052452 0.507229
    1단계 RF + peak_season(11~12월)   0.043981 0.052584 0.482036

원래 1단계 로지스틱 대비 변화:
  1단계 로지스틱 + purchase_month(정수): PR-AUC +0.0002, ROC-AUC -0.0011
  1단계 로지스틱 + peak_season(11~12월): PR-AUC +0.0001, ROC-AUC +0.0011


### 판단 기준 (작성 시점 미증명)

> 아래 "결과 해석 시 주의" 단락은 결과를 본 뒤에 추가한 내용이다. 판단 기준은 결과 확인 전에 정했다고 기록했으나 커밋 이력이 없어 시점은 증명하지 못한다. 또 ±0.01 기준은 1단계 모델 전체의 개선폭(기저율 대비 PR-AUC 약 +0.008)보다 커서 작은 효과는 가려내지 못하므로 이 ablation은 보조 근거 수준이다.

- PR-AUC 또는 ROC-AUC가 **0.01 이상 개선**되면: D-15 미사용 결정이 실제로 신호를 버린 것 — 최소한 "제외 시 성능 손실 폭"을
  한계로 명시하고, 포함 여부를 재검토 대상으로 올린다.
- 개선이 **0.01 미만**이면: 그래프로 본 우려(일회성 스파이크 과적합)가 실제로도 맞았거나, 애초에 이 변수들로는
  설명 안 되는 다른 요인이 있다는 뜻 — D-15 미사용 결정 유지, 이 ablation 결과를 근거로 추가.

### 결과 해석 시 주의

- 테스트 기간이 2018-05~08이라 (B) 성수기 더미(11~12월)는 **테스트에서 전부 0(상수)**이다. 테스트 순위에 새 정보를 줄 수 없으므로,
  (B)의 변화는 학습 구간의 2017-11 스파이크를 흡수한 뒤 다른 계수가 달라진 효과일 뿐이다. 즉 "테스트 구간의 계절 효과는 구조적으로 검증 불가"가 정확한 표현이다.
- 위 기준은 로지스틱 기준이며, RF의 변화(정수 PR-AUC −0.0030/ROC +0.0017, 더미 PR-AUC +0.0007/ROC +0.0008)도 모두 ±0.01 안이다.


## 6. 진단 — 1단계 신호가 왜 약한가 (변수별 단독 AUC, 월별 추이)

목적: 1단계가 기저율을 못 넘은 이유가 **(가) 변수 자체에 신호가 없음**인지, **(나) 학습 구간에서만 있던 신호(시기 효과)가 테스트에서 사라짐**인지 가른다.
읽는 법:
- 단독 AUC 0.5 = 무작위, 0.5 미만 = 방향이 반대. 학습·테스트 둘 다 ~0.5 → (가), 학습만 높고 테스트 ~0.5 → (나) 쪽.
- 월별 AUC: 같은 달 안에서 비교하므로 "그 달에 지연이 많았다"는 시기 효과가 빠진다. 월별 AUC도 ~0.5면 합산 AUC의 신호는 시기 효과였다는 뜻.
- 이 절은 **원인 설명까지만** 한다(성능 개선 시도는 하지 않음). 결과를 보고 판단 기준과 해석을 아래 markdown에 기록한다.

6-1~6-3은 1차 진단, 6-4~6-8은 1차 결과를 보고 추가한 확인이다(6-4: 월별 양성 건수와 가중 평균, 6-5: fallback 값의 시간 변화, 6-6: 2018-08 distance 역전 원인, 6-7: fallback 행만의 월 내 AUC, 6-8: 예상소요일·주 단위 late율, 6-9: fallback 값 vs 실제 누적 late율, 6-10: 8월 distance 역전 vs 우측절단 — 거리 구간별, 6-11: 7월 vs 8/1~14 예상일·실제 소요일 분포).


In [12]:
# 6-1. 변수별 단독 ROC-AUC (학습 vs 테스트)
def univariate_auc(df, cols, target):
    out = {}
    for c in cols:
        s = df[[c, target]].dropna()
        if s[c].nunique() < 2 or s[target].nunique() < 2:
            continue
        out[c] = roc_auc_score(s[target].astype(int), s[c])
    return pd.Series(out)

uni_cols_1 = STAGE1_NUM + STAGE1_BIN
auc_tbl_1 = pd.DataFrame({
    "학습": univariate_auc(master[train_mask], uni_cols_1, "late"),
    "테스트": univariate_auc(master[test_mask], uni_cols_1, "late"),
})
print("[1단계 late] 변수별 단독 ROC-AUC")
print(auc_tbl_1.round(4))

uni_cols_2 = STAGE1_NUM + STAGE2_NUM + STAGE1_BIN
auc_tbl_2 = pd.DataFrame({
    "학습": univariate_auc(master[stage2_mask_train], uni_cols_2, "low"),
    "테스트": univariate_auc(master[stage2_mask_test], uni_cols_2, "low"),
})
print("\n[2단계 low] 변수별 단독 ROC-AUC")
print(auc_tbl_2.round(4))


[1단계 late] 변수별 단독 ROC-AUC
                                  학습     테스트
distance_km_filled            0.5980  0.4738
seller_avg_delay              0.5947  0.5194
dist_missing                  0.5006  0.5028
seller_avg_delay_is_fallback  0.4863  0.4910

[2단계 low] 변수별 단독 ROC-AUC
                                  학습     테스트
distance_km_filled            0.5481  0.5213
seller_avg_delay              0.5513  0.5280
price_sum                     0.5419  0.5493
freight_sum                   0.5925  0.5570
installments_max              0.5253  0.5273
dist_missing                  0.4994  0.5006
seller_avg_delay_is_fallback  0.4923  0.4885


In [13]:
# 6-2. seller_avg_delay: fallback / non-fallback 분리 (D-01의 0.6236 vs 0.5855는 학습 구간 기준으로 기록된 값)
rows = []
for flag in [0, 1]:
    for split_name, mask in [("학습", train_mask), ("테스트", test_mask)]:
        sub = master[mask & (master["seller_avg_delay_is_fallback"] == flag)]
        if len(sub) == 0 or sub["late"].nunique() < 2:
            continue
        rows.append({"is_fallback": flag, "구간": split_name, "n": len(sub),
                     "late율": sub["late"].mean(),
                     "AUC(seller_avg_delay)": roc_auc_score(sub["late"].astype(int), sub["seller_avg_delay"])})
print(pd.DataFrame(rows).round(4).to_string(index=False))


 is_fallback  구간     n  late율  AUC(seller_avg_delay)
           0  학습 46806 0.0794                 0.5855
           0 테스트 18232 0.0450                 0.5323
           1  학습 24046 0.0707                 0.6236
           1 테스트  7120 0.0413                 0.4426


In [14]:
# 6-3. 월별 추이 + 월별(같은 달 안) 단독 AUC — 시기 효과 제거 후에도 신호가 남는가
ym = master["purchase_ts"].dt.to_period("M")
monthly = master.groupby(ym).agg(
    n=("late", "size"),
    late_rate=("late", "mean"),
    mean_seller_avg_delay=("seller_avg_delay", "mean"),
    fallback_rate=("seller_avg_delay_is_fallback", "mean"),
)

def month_auc(g, col):
    if g["late"].nunique() < 2 or g[col].nunique() < 2:
        return np.nan
    return roc_auc_score(g["late"].astype(int), g[col])

monthly["AUC_seller_avg_delay(월 내)"] = master.groupby(ym).apply(lambda g: month_auc(g, "seller_avg_delay"))
monthly["AUC_distance(월 내)"] = master.groupby(ym).apply(lambda g: month_auc(g, "distance_km_filled"))
monthly["구간"] = np.where(monthly.index < pd.Period("2018-05"), "학습", "테스트")
print(monthly.round(4).to_string())

print("\n월 내 AUC 평균(구간별):")
print(monthly.groupby("구간")[["AUC_seller_avg_delay(월 내)", "AUC_distance(월 내)"]].mean().round(4))

# 월별 평균 seller_avg_delay와 월별 late율의 상관 (시기 동행 정도)
for g_name, g in monthly.groupby("구간"):
    print(f"{g_name}: corr(월평균 seller_avg_delay, 월 late율) = {g['mean_seller_avg_delay'].corr(g['late_rate']):.3f} (개월 수 {len(g)})")


                n  late_rate  mean_seller_avg_delay  fallback_rate  AUC_seller_avg_delay(월 내)  AUC_distance(월 내)   구간
purchase_ts                                                                                                          
2017-01       750     0.0293                 0.0000         0.9907                        NaN             0.5897   학습
2017-02      1653     0.0296                 0.0001         0.9274                     0.5272             0.4410   학습
2017-03      2546     0.0456                 0.0053         0.7109                     0.4895             0.6435   학습
2017-04      2303     0.0656                 0.0165         0.5232                     0.5228             0.5317   학습
2017-05      3545     0.0299                 0.0268         0.4677                     0.4881             0.3769   학습
2017-06      3135     0.0303                 0.0286         0.3566                     0.5287             0.5110   학습
2017-07      3872     0.0279                 0.0281     

In [15]:
# 6-4. 월별 양성 건수 + 양성 수 가중 평균 (표본이 작은 달의 영향을 줄임)
# MIN_POS: 월 내 AUC 평균에 넣을 최소 양성 건수. 1차 결과를 본 뒤 정한 진단용 기준이므로 보고서에는 그렇게 적을 것.
MIN_POS = 100

ym = master["purchase_ts"].dt.to_period("M")

def auc_or_nan(g, col):
    if g["late"].nunique() < 2 or g[col].nunique() < 2:
        return np.nan
    return roc_auc_score(g["late"].astype(int), g[col])

rows = []
for p, g in master.groupby(ym):
    nf = g[g["seller_avg_delay_is_fallback"] == 0]
    rows.append({
        "월": str(p),
        "구간": "학습" if p < pd.Period("2018-05") else "테스트",
        "n": len(g),
        "n_pos": int(g["late"].sum()),
        "n_pos_nonfb": int(nf["late"].sum()),
        "AUC_seller": auc_or_nan(g, "seller_avg_delay"),
        "AUC_seller_nonfb": auc_or_nan(nf, "seller_avg_delay"),
        "AUC_dist": auc_or_nan(g, "distance_km_filled"),
    })
mt = pd.DataFrame(rows)
mt["표본작음(<MIN_POS)"] = np.where(mt["n_pos"] < MIN_POS, "*", "")
print(mt.round(4).to_string(index=False))

def wavg(df, col, wcol):
    d = df.dropna(subset=[col])
    d = d[d[wcol] >= MIN_POS]
    if len(d) == 0:
        return np.nan, 0
    return np.average(d[col], weights=d[wcol]), len(d)

print(f"\n월 내 AUC: 양성 {MIN_POS}건 이상인 달만, 양성 수 가중 평균 (괄호는 사용한 개월 수)")
for seg, g in mt.groupby("구간"):
    out = []
    for col, w in [("AUC_seller", "n_pos"), ("AUC_seller_nonfb", "n_pos_nonfb"), ("AUC_dist", "n_pos")]:
        v, k = wavg(g, col, w)
        out.append(f"{col}={v:.4f}({k})")
    print(f"  {seg}: " + "  ".join(out))


      월  구간    n  n_pos  n_pos_nonfb  AUC_seller  AUC_seller_nonfb  AUC_dist 표본작음(<MIN_POS)
2017-01  학습  750     22            0         NaN               NaN    0.5897              *
2017-02  학습 1653     49            2      0.5272               NaN    0.4410              *
2017-03  학습 2546    116           39      0.4895            0.5038    0.6435               
2017-04  학습 2303    151           59      0.5228            0.4991    0.5317               
2017-05  학습 3545    106           61      0.4881            0.5218    0.3769               
2017-06  학습 3135     95           51      0.5287            0.5335    0.5110              *
2017-07  학습 3872    108           73      0.5264            0.5439    0.6135               
2017-08  학습 4193    122           78      0.5130            0.5220    0.5724               
2017-09  학습 4150    182          125      0.5163            0.5467    0.6083               
2017-10  학습 4478    187          123      0.5738            0.5852    0.5549    

In [16]:
# 6-5. fallback 행의 seller_avg_delay는 상수인가, 시간에 따라 변하는가
# 월 안에서 거의 상수(nunique 작음)이고 월마다 평균이 달라지면 → fallback 값이 시기 대리변수로 작동했을 가능성
fb = master[master["seller_avg_delay_is_fallback"] == 1]
fb_m = fb.groupby(fb["purchase_ts"].dt.to_period("M"))["seller_avg_delay"].agg(
    n="size", nunique="nunique", mean="mean", std="std", min="min", max="max")
print("[fallback 행] 월별 seller_avg_delay")
print(fb_m.round(4).to_string())

late_m = master.groupby(master["purchase_ts"].dt.to_period("M"))["late"].mean().rename("late_rate")
cmp_ = fb_m[["mean"]].join(late_m, how="inner")
tr = cmp_.index < pd.Period("2018-05")
print(f"\nfallback 월평균 vs 월 late율 상관: 학습 {cmp_[tr]['mean'].corr(cmp_[tr]['late_rate']):.3f} (개월 {tr.sum()}), "
      f"테스트 {cmp_[~tr]['mean'].corr(cmp_[~tr]['late_rate']):.3f} (개월 {(~tr).sum()}, 참고용)")
print("월 내 std 평균:", round(fb_m["std"].mean(), 4), "/ 월 간 평균의 std:", round(fb_m["mean"].std(), 4))


[fallback 행] 월별 seller_avg_delay
                n  nunique    mean     std     min     max
purchase_ts                                               
2017-01       743        1  0.0000  0.0000  0.0000  0.0000
2017-02      1533       12  0.0001  0.0002  0.0000  0.0006
2017-03      1810      714  0.0066  0.0035  0.0006  0.0134
2017-04      1205      463  0.0186  0.0033  0.0129  0.0237
2017-05      1658      789  0.0284  0.0037  0.0237  0.0342
2017-06      1118      506  0.0332  0.0005  0.0322  0.0341
2017-07      1218      586  0.0318  0.0003  0.0311  0.0324
2017-08      1427      736  0.0297  0.0004  0.0293  0.0311
2017-09      1287      677  0.0298  0.0009  0.0290  0.0314
2017-10      1345      707  0.0322  0.0006  0.0309  0.0331
2017-11      2209     1011  0.0332  0.0003  0.0324  0.0337
2017-12      1491      961  0.0352  0.0028  0.0333  0.0431
2018-01      1939     1103  0.0488  0.0024  0.0430  0.0510
2018-02      1623      915  0.0500  0.0002  0.0497  0.0507
2018-03      1662      

In [17]:
# 6-6. 2018-08 distance 역전 원인 탐색 — 8월 distance 구간별 late율, 고객·셀러 주 쏠림
# 원인을 단정하지 않고 어디에 쏠려 있는지만 본다.
bins = [0, 100, 300, 600, 1000, 1500, 5000]
dbin = pd.cut(master["distance_km_filled"], bins=bins, include_lowest=True)
periods = {
    "학습(~2018-04)": train_mask,
    "2018-05": ym == pd.Period("2018-05"),
    "2018-06": ym == pd.Period("2018-06"),
    "2018-07": ym == pd.Period("2018-07"),
    "2018-08": ym == pd.Period("2018-08"),
}
late_tbl, n_tbl = {}, {}
for name, m in periods.items():
    g = master[m].groupby(dbin[m], observed=True)["late"].agg(["mean", "size"])
    late_tbl[name], n_tbl[name] = g["mean"], g["size"]
print("[distance 구간별 late율]")
print(pd.DataFrame(late_tbl).round(4).to_string())
print("\n[distance 구간별 주문 수]")
print(pd.DataFrame(n_tbl).to_string())

aug = master[ym == pd.Period("2018-08")]
ref = master[(ym >= pd.Period("2018-05")) & (ym <= pd.Period("2018-07"))]

def state_table(col, top=8):
    if col not in master.columns:
        print(f"\n{col} 컬럼이 master_features에 없어 건너뜀")
        return
    a = aug.groupby(col).agg(n_aug=("late", "size"), pos_aug=("late", "sum"),
                             late_aug=("late", "mean"), dist_aug=("distance_km_filled", "mean"))
    r = ref.groupby(col).agg(late_ref=("late", "mean"))
    out = a.join(r).sort_values("pos_aug", ascending=False).head(top)
    out["양성비중"] = out["pos_aug"] / aug["late"].sum()
    print(f"\n[2018-08 {col}별 (양성 많은 순 상위 {top}), late_ref = 2018-05~07 같은 주 late율]")
    print(out.round(4).to_string())

state_table("customer_state")
state_table("seller_state")

if {"customer_state", "seller_state"} <= set(master.columns):
    same = (master["customer_state"] == master["seller_state"]).rename("같은주")
    for name, m in [("2018-05~07", (ym >= pd.Period("2018-05")) & (ym <= pd.Period("2018-07"))),
                    ("2018-08", ym == pd.Period("2018-08"))]:
        g = master[m].groupby(same[m])["late"].agg(["mean", "size"])
        print(f"\n[{name}] 고객·셀러 같은 주 여부별 late율")
        print(g.round(4).to_string())


[distance 구간별 late율]
                    학습(~2018-04)  2018-05  2018-06  2018-07  2018-08
distance_km_filled                                                  
(-0.001, 100.0]           0.0372   0.0387   0.0096   0.0513   0.1319
(100.0, 300.0]            0.0559   0.0613   0.0067   0.0197   0.0744
(300.0, 600.0]            0.0799   0.0664   0.0057   0.0205   0.0312
(600.0, 1000.0]           0.0835   0.0567   0.0112   0.0215   0.0382
(1000.0, 1500.0]          0.1008   0.1045   0.0338   0.0475   0.0110
(1500.0, 5000.0]          0.1335   0.1300   0.0303   0.0670   0.0189

[distance 구간별 주문 수]
                    학습(~2018-04)  2018-05  2018-06  2018-07  2018-08
distance_km_filled                                                  
(-0.001, 100.0]            12209     1423     1253     1267     1600
(100.0, 300.0]              9707     1044      900      812      900
(300.0, 600.0]             23609     2152     1926     1850     2018
(600.0, 1000.0]            13731     1163     1071     1116  

### 6-7~6-11 추가 확인 (6-4~6-6 결과를 본 뒤 추가; 6-9~6-11은 그 뒤 점검 과정에서 추가)
- 6-7: fallback 값이 시기 대리변수라는 추정을 같은 달 안 AUC로 직접 확인
- 6-8: 2018-08 distance 역전의 원인 후보(예상소요일 단축, 8월 초 집중)와 8월 말 데이터 절단 확인
- 6-9: fallback 값 vs 구매순서 실제 누적 late율(진단용)
- 6-10: 8월 distance 역전을 8/1~14 · 8/15~31 · 5~7월로 나눠 우측절단(04 D-17)과 비교
- 6-11: 8월 초 지연 급증이 예상일 쪽 변화인지 실제 배송 쪽 변화인지 7월 vs 8/1~14 분포로 비교

In [18]:
# 6-7. fallback 행만의 월 내 AUC — "fallback 값은 시기 대리변수" 추정을 직접 확인
# 같은 달 안에서 fallback 행끼리 비교하므로 시기 효과가 빠진다. 월 내 AUC가 0.5 근처면 fallback의 합산 AUC(학습 0.624)는 시기 효과로 볼 수 있다.
# MIN_POS(6-4)는 결과를 본 뒤 정한 진단용 기준이므로 같은 기준을 쓰되 보고서에 그렇게 적을 것.
rows = []
for p, g in master.groupby(ym):
    f = g[g["seller_avg_delay_is_fallback"] == 1]
    rows.append({"월": str(p), "구간": "학습" if p < pd.Period("2018-05") else "테스트",
                 "n_fb": len(f), "n_pos_fb": int(f["late"].sum()),
                 "AUC_seller_fb": auc_or_nan(f, "seller_avg_delay")})
mfb = pd.DataFrame(rows)
mfb["표본작음(<MIN_POS)"] = np.where(mfb["n_pos_fb"] < MIN_POS, "*", "")
print(mfb.round(4).to_string(index=False))
print(f"\nfallback 행 월 내 AUC: 양성 {MIN_POS}건 이상인 달만, 양성 수 가중 평균 (괄호는 개월 수)")
for seg, g in mfb.groupby("구간"):
    v, k = wavg(g, "AUC_seller_fb", "n_pos_fb")
    print(f"  {seg}: AUC_seller_fb={v:.4f}({k})")


      월  구간  n_fb  n_pos_fb  AUC_seller_fb 표본작음(<MIN_POS)
2017-01  학습   743        22            NaN              *
2017-02  학습  1533        47         0.5262              *
2017-03  학습  1810        77         0.5180              *
2017-04  학습  1205        92         0.5146              *
2017-05  학습  1658        45         0.4340              *
2017-06  학습  1118        44         0.4545              *
2017-07  학습  1218        35         0.4441              *
2017-08  학습  1427        44         0.4935              *
2017-09  학습  1287        57         0.3762              *
2017-10  학습  1345        64         0.5608              *
2017-11  학습  2209       289         0.5979               
2017-12  학습  1491       105         0.4795               
2018-01  학습  1939       128         0.6105               
2018-02  학습  1623       238         0.5744               
2018-03  학습  1662       309         0.3898               
2018-04  학습  1778       105         0.4242               
2018-05 테스트  1

In [19]:
# 6-8. 8월 원인 후보: (a) 예상소요일(예상도착일 - 구매일)이 짧아졌는가, (b) 8월 안에서 주 단위로 몰렸는가
# 원인을 단정하지 않고 값만 본다. late가 예상도착일 기준이라 예상일 설정이 바뀌어도 지연이 늘어 보인다.
# order_estimated_delivery_date는 EDA 전용(모델 입력 금지). 컬럼이 없으면 건너뛴다.
need = {"order_estimated_delivery_date", "customer_state", "seller_state"}
if not need <= set(master.columns):
    print("필요 컬럼이 master_features에 없어 건너뜀:", need - set(master.columns))
else:
    d = master.copy()
    d["est_days"] = (d["order_estimated_delivery_date"] - d["purchase_ts"]).dt.total_seconds() / 86400
    d["act_days"] = (d["order_delivered_customer_date"] - d["purchase_ts"]).dt.total_seconds() / 86400 \
        if "order_delivered_customer_date" in d.columns else np.nan
    d["ym"] = d["purchase_ts"].dt.to_period("M")
    d["sp_sp"] = (d["customer_state"] == "SP") & (d["seller_state"] == "SP")

    print("[SP->SP (고객 SP & 대표 셀러 SP)] 월별 late율·예상소요일·실제소요일 (일)")
    t = d[d["sp_sp"] & (d["ym"] >= pd.Period("2018-01"))].groupby("ym").agg(
        n=("late", "size"), late=("late", "mean"), est_days=("est_days", "mean"), act_days=("act_days", "mean"))
    print(t.round(3).to_string())

    print("\n[그 외] 월별")
    t2 = d[~d["sp_sp"] & (d["ym"] >= pd.Period("2018-01"))].groupby("ym").agg(
        n=("late", "size"), late=("late", "mean"), est_days=("est_days", "mean"), act_days=("act_days", "mean"))
    print(t2.round(3).to_string())

    a = d[d["sp_sp"] & (d["ym"] == pd.Period("2018-08"))].copy()
    a["구매주"] = a["purchase_ts"].dt.to_period("W")
    print("\n[2018-08 SP->SP] 구매 주별")
    print(a.groupby("구매주").agg(n=("late", "size"), late=("late", "mean"),
                                  est_days=("est_days", "mean"), act_days=("act_days", "mean")).round(3).to_string())
    if "order_delivered_customer_date" in a.columns:
        print("\n[2018-08 SP->SP 지연 주문] 도착 주별 건수")
        print(a.loc[a["late"] == 1, "order_delivered_customer_date"].dt.to_period("W").value_counts().sort_index().to_string())

    # 데이터 끝 절단: 8월 말 일별 주문 수
    print("\n[2018-08 말 일별 주문 수 (base 기준)]")
    print(d[d["ym"] == pd.Period("2018-08")].groupby(d["purchase_ts"].dt.date).size().tail(8).to_string())


[SP->SP (고객 SP & 대표 셀러 SP)] 월별 late율·예상소요일·실제소요일 (일)
            n   late  est_days  act_days
ym                                      
2018-01  2276  0.030    19.957     8.911
2018-02  1891  0.079    18.468    10.119
2018-03  2319  0.088    15.590     9.292
2018-04  2359  0.015    17.422     6.583
2018-05  2461  0.043    16.424     7.292
2018-06  2068  0.008    19.544     5.870
2018-07  2040  0.041    12.927     6.063
2018-08  2481  0.104     9.230     5.678

[그 외] 월별
            n   late  est_days  act_days
ym                                      
2018-01  4793  0.070    29.316    16.536
2018-02  4665  0.167    26.984    19.715
2018-03  4684  0.240    25.223    19.772
2018-04  4439  0.061    27.005    14.112
2018-05  4288  0.078    26.603    13.788
2018-06  4028  0.014    31.997    10.969
2018-07  4116  0.030    23.042    10.394
2018-08  3870  0.035    19.005     9.050

[2018-08 SP->SP] 구매 주별
                         n   late  est_days  act_days
구매주                                    

In [20]:
# 6-9. fallback 값(배송완료분만 쓴 누적 지연비율) vs 실제 누적 late율(구매순서, 배송 여부 무관) — 월별 비교
# 진단용: 구매순서 누적 late율은 아직 배송 전인 주문의 라벨을 쓰므로 모델 입력으로는 쓰지 않는다(05 2-0의 누출 방식과 같음).
# 가설: 배송완료분만 쓰면 지연 주문이 늦게 반영돼 초반 fallback 값이 실제보다 낮다. 격차가 초반에 크고 시간이 지나며 줄면 가설과 부합.
m = master.sort_values(["purchase_ts", "order_id"]).reset_index(drop=True)
m["true_cum_late"] = (m["late"].cumsum() - m["late"]) / np.arange(len(m)).clip(min=1)
m.loc[0, "true_cum_late"] = np.nan
mm = m["purchase_ts"].dt.to_period("M")
isfb = m["seller_avg_delay_is_fallback"] == 1
tbl = pd.DataFrame({"fallback_평균": m[isfb].groupby(mm[isfb])["seller_avg_delay"].mean(),
                    "실제_누적_late율": m.groupby(mm)["true_cum_late"].mean(),
                    "월_late율": m.groupby(mm)["late"].mean()})
tbl["격차(실제-fallback)"] = tbl["실제_누적_late율"] - tbl["fallback_평균"]
print(tbl.round(4).to_string())


             fallback_평균  실제_누적_late율  월_late율  격차(실제-fallback)
purchase_ts                                                    
2017-01           0.0000       0.0267   0.0293           0.0267
2017-02           0.0001       0.0272   0.0296           0.0271
2017-03           0.0066       0.0339   0.0456           0.0272
2017-04           0.0186       0.0425   0.0656           0.0239
2017-05           0.0284       0.0440   0.0299           0.0156
2017-06           0.0332       0.0395   0.0303           0.0063
2017-07           0.0318       0.0373   0.0279           0.0056
2017-08           0.0297       0.0355   0.0291           0.0058
2017-09           0.0298       0.0359   0.0439           0.0061
2017-10           0.0322       0.0364   0.0418           0.0043
2017-11           0.0332       0.0429   0.1240           0.0098
2017-12           0.0352       0.0563   0.0746           0.0211
2018-01           0.0488       0.0558   0.0570           0.0070
2018-02           0.0500       0.0594   

In [21]:
# 6-10. 2018-08 distance 역전 vs 우측절단 — 8/1~14, 8/15~31, 5~7월을 나눠서 본다 (04 노트북 D-17 연결)
# 원인을 단정하지 않고 거리 구간별 late율·구성비·AUC만 본다. distance_km_filled 사용(미매칭 477건은 주 평균으로 채워진 값).
ref = master[(master["purchase_ts"] >= "2018-05-01") & (master["purchase_ts"] < "2018-08-01")].copy()
ref["구간"] = "5~7월"
aug = master[(master["purchase_ts"] >= "2018-08-01") & (master["purchase_ts"] < "2018-09-01")].copy()
aug["구간"] = np.where(aug["purchase_ts"].dt.day >= 15, "8/15~31", "8/1~14")
dd = pd.concat([ref, aug])
dd["거리구간"] = pd.cut(dd["distance_km_filled"], bins=[-0.1, 100, 300, 600, 1000, np.inf],
                      labels=["0~100", "100~300", "300~600", "600~1000", "1000+"])
print("[거리 구간별 late율 / 건수]")
print(dd.groupby(["거리구간", "구간"], observed=True).agg(n=("late", "size"), late=("late", "mean"))
        .unstack("구간").round(3).to_string())
print("\n[구간별 거리 구간 구성비 — 생존표본 편향이면 8/15~31에서 먼 거리 비중이 줄어든다]")
print(dd.groupby("구간")["거리구간"].value_counts(normalize=True).unstack().round(3).to_string())
print("\n[구간별 distance 단독 AUC (양성 수가 적으면 해석 약함)]")
for k, g in dd.groupby("구간"):
    print(f"  {k}: AUC={roc_auc_score(g['late'], g['distance_km_filled']):.4f} / n={len(g)} / 양성={int(g['late'].sum())}")

[거리 구간별 late율 / 건수]
             n                  late               
구간        5~7월 8/15~31 8/1~14   5~7월 8/15~31 8/1~14
거리구간                                               
0~100     3943     727    873  0.033   0.050  0.200
100~300   2756     361    539  0.031   0.044  0.095
300~600   5928     754   1264  0.032   0.005  0.047
600~1000  3350     372    675  0.030   0.005  0.056
1000+     3024     200    586  0.069   0.000  0.020

[구간별 거리 구간 구성비 — 생존표본 편향이면 8/15~31에서 먼 거리 비중이 줄어든다]
거리구간     0~100  100~300  300~600  600~1000  1000+
구간                                               
5~7월     0.208    0.145    0.312     0.176  0.159
8/15~31  0.301    0.150    0.312     0.154  0.083
8/1~14   0.222    0.137    0.321     0.171  0.149

[구간별 distance 단독 AUC (양성 수가 적으면 해석 약함)]
  5~7월: AUC=0.5681 / n=19001 / 양성=722
  8/15~31: AUC=0.2475 / n=2414 / 양성=58
  8/1~14: AUC=0.2959 / n=3937 / 양성=335


### 6절 해석

**1차 진단(6-1~6-3)**
- 단독 AUC(학습→테스트): distance 0.598→0.474(방향 역전), seller_avg_delay 0.595→0.519, dist_missing·is_fallback은 양쪽 ~0.5. 2단계 변수는 0.52~0.56에서 학습·테스트가 비슷함(price_sum 0.542→0.549, freight_sum 0.593→0.557).
- fallback 분리: non-fallback 0.586→0.532(유지), fallback 0.624→0.443(역전).
- 월 내 AUC 단순 평균은 표본 작은 달에 흔들려 6-4 가중 평균으로 대체(월별 상관은 테스트 4개월이라 해석 불가).

**추가 확인(6-4~6-6)**
- 6-4 (양성 100건 이상 달, 양성 수 가중): seller_avg_delay 학습 0.548(13개월) → 테스트 0.529(3개월), non-fallback만 0.561(8) → 0.545(3). distance 학습 0.608(13) → 테스트 0.469(3). 셀러 변수는 약하지만 시기를 걷어내도 남고 테스트에서도 유지됨. distance는 학습에서 월 내 신호가 뚜렷했으나 테스트에서 역전(2018-05 0.591, 07 0.516, 08 0.307).
- 6-5 fallback 행의 seller_avg_delay는 상수가 아니라 월별로 계단식 상승: 2017-01 0.000 → 2017-06 0.033 → 2018-01 0.049 → 2018-05 0.072. 월 내 표준편차 평균 0.0014 vs 월 간 평균의 표준편차 0.0231로 거의 월 단위 시간 값. 학습 구간 월평균과 월 late율 상관 0.465. 월평균과 late율의 상관은 16개월 기준이라 약한 근거임. 따라서 fallback 값이 시간 대리변수로 작동해 학습 AUC 0.624를 만들었다는 것은 '그렇게 보인다'는 추정이었고, 6-7(fallback 행만의 월 내 AUC)로 확인했다(아래 6-7 항목). 테스트 구간 fallback 값은 평평하지 않고 0.0723 → 0.0725 → 0.0692 → 0.0672로 내려가며 월 안의 nunique도 873~1,151임(상수가 아님). 테스트에서 late율이 내려간 것과 맞물려 AUC가 0.443으로 뒤집힌 것으로 보인다. 테스트 구간 값이 움직이는 이유는 05 확인으로 풀렸다: fallback 값은 '구매 시점 이전에 배송이 끝난 주문'의 전체 기간 누적 지연비율이라, 테스트 기간에 구매 시점 이전에 이미 배송이 끝난 주문의 late가 반영되며 움직인다(구매 시점에 알 수 있는 라벨이므로 누수 아님).
- 6-6 2018-08 distance 역전: 학습·2018-05는 거리가 멀수록 late율이 높음(0~100km 약 3.7~3.9% → 1,500km+ 약 13%). 2018-08은 반대로 0~100km 13.2%, 100~300km 7.4% → 1,000km 이상 1~2%. 8월 양성 393건 중 72.5%가 고객 SP, 80.7%가 셀러 SP. SP 고객 late율 9.0%(2018-05~07은 3.0%). 고객·셀러 같은 주 late율이 2018-08 10.2%(n=2,829) vs 다른 주 3.0%(2018-05~07은 같은 주 3.1% vs 다른 주 4.2%). 8월 지연이 SP 내(근거리) 배송에 몰려 거리-지연 관계가 뒤집힘. 원인(운송·예상일 설정·기타)은 이 자료로는 모름.

**결론((가) 변수 한계 / (나) 시기 효과)**: 둘 다 해당.
- (나) 시기 효과: fallback 행의 학습 합산 AUC 0.624는 같은 달 안에서 비교하면 0.514(6-7)로 0.5 근처까지 내려가, 대부분 fallback 값의 시간 상승이 만든 것으로 판단. distance는 학습에서는 실제 신호였으나 2018-08 SP 내 지연 급증으로 테스트에서 관계가 뒤집힘(분포 변화).
- (가) 변수 한계: 시기를 걷어낸 seller_avg_delay 신호도 약함(월 내 AUC 0.53~0.56).
- 1단계가 기저율 수준에 머문 직접 원인은 "약한 신호 + 테스트 구간 중 8월 하나에서 distance 관계가 반대로 바뀜"의 조합. 테스트 4개월이라 이 한 번의 변화가 결과를 크게 좌우함(7-9: 8월 제외 시 로지스틱 1.19→1.57배, RF 1.04→1.32배, R1 1.05→1.41배).
- 8월 초 근거리 지연 급증은 예상소요일 단축(7월 대비 중앙값 약 5일)과 8/1~7의 실제 소요일 증가가 겹쳐 나타남(6-11). late 라벨이 예상일에 의존하므로 라벨 기준의 시간 일관성이 1단계의 추가 한계이며, 예상일을 왜 줄였는지와 첫 주에 실제로 느려진 원인은 모름.

**한계·주의**
- 테스트 구간은 D-03대로 유지. 2018-08 제외 결과 등은 민감도 분석으로만 보고(테스트 구간 변경 아님).
- fallback 값의 정의(05 확인): 구매시각 이전에 배송이 끝난 주문의 전역 지연비율(시간순 누적). D-01의 기존 문구(학습 구간 전체 평균, 학습 행은 자기 제외)와 달라 D-01·data_dictionary 문구를 이 정의로 고쳐야 함. 맨 앞 60건만 첫 유효값으로 채워 미래값을 쓰는데 영향은 미미함.
- 추정(미검증): '배송이 끝난 주문만 쓰는' 규칙은 지연 주문이 늦게 배송되므로 특히 초반 값을 낮게 편향시킬 수 있고, 그렇다면 fallback 값의 계단식 상승 일부는 실제 late율 상승이 아니라 이 구조에서 나온 것일 수 있음. 6-9로 확인: 초반(2017-01~05)과 late 급증 직후(2017-12, 2018-03)에 fallback이 실제 누적보다 낮아 가설과 부합(지연 주문이 늦게 반영되는 시차). 테스트 구간에서는 격차가 거의 없음. 이 격차는 진단용 구매순서 누적과의 비교이므로 모델 입력을 바꾸는 근거로 쓰지 않고 한계로만 기록.
- 월 내 AUC 가중 평균은 학습 13개월 대 테스트 3개월(non-fallback 8 대 3, fallback 6 대 2)로 비대칭이라 비교 시 같이 적을 것. 6-7 테스트 0.454는 2018-05(0.550)와 08(0.360) 두 달 평균이라 방향을 말하기에는 약함.
- 월 내 AUC 가중 평균은 MIN_POS=100을 결과를 본 뒤 정한 진단용 기준임.

**강사님 보고에 넣을 문장**: "1단계 모델은 기저율 수준(PR-AUC 0.052 vs 0.044)에 그쳤다. 원인 진단 결과 (1) seller_avg_delay의 학습 신호 일부는 fallback 값이 시간에 따라 오르며 생긴 시기 효과이고(fallback 행끼리 같은 달 안에서 비교하면 AUC 0.51), 시기를 걷어낸 신호는 약하며(월 내 AUC 0.53~0.56), (2) distance는 학습에서는 월 내 AUC 0.61이었으나 테스트의 2018-08에 지연이 상파울루 내 근거리 배송에 몰리면서 관계가 뒤집혔다(월 내 AUC 0.31). 같은 시기 상파울루 내 예상소요일 중앙값이 7월 13.1일에서 8월 첫 두 주 약 8일로 5일 가까이 짧아졌고(두 주 공통), 실제 소요일은 첫 주(8/1~7)에만 늘었다(6-11). 예상일 단축이 지연에 취약하게 만든 조건으로 작용했을 가능성이 크고(실제 소요일이 7월 수준이던 둘째 주에도 late 4% → 8%), 첫 주(late 22%)에는 실제 지연도 겹쳤다. 예상일을 왜 줄였는지, 첫 주에 실제로 느려진 원인은 이 자료로 확정할 수 없다. 따라서 변수 중요도로 지연 요인을 설명하기보다, 월별·지역별 기술통계로 요인을 설명하고 시기 변화를 한계로 명시하려 한다."

**추가 확인 결과 (6-7, 6-8)**
- 6-7 fallback 행만의 월 내 AUC(양성 100건 이상 달, 가중): 학습 0.514(6개월: 2017-11~2018-04), 테스트 0.454(2개월: 2018-05 0.550, 08 0.360). 합산 AUC 0.624가 같은 달 안에서는 0.5 근처라 시기 효과라는 추정이 맞음. 학습 6개월의 월별 값은 0.39~0.61로 방향 없이 흔들림. 테스트 0.454는 두 달뿐이라 참고용.
- 6-9 fallback 값 vs 실제 누적 late율(구매순서, 진단용): 격차(실제−fallback)가 2017-01~04에 0.024~0.027(fallback 0.000~0.019 vs 실제 0.027~0.043), 2017-06~11에 0.004~0.010으로 줄었다가, late 급증 직후인 2017-12(0.021)와 2018-03(0.022)에 다시 벌어지고, 테스트 구간(2018-05~08)에는 0.0002~0.0035로 거의 일치. 실제 누적 late율 자체도 0.027(2017-01) → 0.079(2018-04)로 올라 fallback 상승의 대부분은 실제 late율 상승이지만, 2017-01~05의 0 → 0.028 상승은 같은 기간 실제 누적이 0.027~0.044로 거의 평평했던 것에 비해 커서 초반 상승은 대부분 '배송완료분만 쓰는' 규칙의 인공물로 보임.
- 6-8 예상소요일(상파울루→상파울루, 대표 셀러 기준): 2018-05 16.4일, 06 19.5일, 07 12.9일, 08 9.2일(01월 20.0일). 실제 소요일은 08 5.7일로 06~07(5.9~6.1일)과 비슷. late는 도착일이 예상도착일보다 늦은 것이라 예상일 길이가 아니라 '예상일 − 실제 소요일'(여유)로 봐야 함. SP→SP 여유는 5월 9.1일, 6월 13.7일, 7월 6.9일, 8월 3.6일로 8월에 가장 작고 late(10.4%)가 가장 높음. 그 외 지역의 8월 여유는 약 10일(19.0 − 9.1)이라 late가 3.5%로 낮은 것이 자연스러움(이전의 "예상일이 가장 짧은데 late 3.5%라 설명 안 됨"은 여유를 보지 않은 비교라 철회). 여유와 late는 월별로 대체로 반비례하지만 엄밀한 단조는 아님(2월 여유 8.4일 late 7.9% vs 7월 6.9일 4.1%). 분해는 6-11 참고.
- 6-8 주 단위(8월 상파울루→상파울루): 구매 주별 late율 17.5% → 16.3% → 6.8% → 3.2%(3~4주차의 낮은 값은 우측절단 영향이 섞일 수 있어 '후반 개선'으로 읽지 않음, 아래 6-10 참고). 첫 2주 실제 소요일 6.7·6.2일, 3~4주차 5.8·4.6일. 지연 258건 중 120건(46.5%)이 8/13~19 주에 도착. 원인(운송·예상일 설정·기타)은 이 자료로는 모름.
- 8월 말 데이터 절단: base 일별 주문이 8/24 96건 → 8/27 66 → 8/28 39 → 8/29 11건이고 8/30~31은 0건. 마지막 주(8/27~)는 n=97, 실제 소요일 2.1일, late 0%로 빠르게 처리된 주문만 남은 선택 효과일 수 있으나 8월 전체의 1.5%라 영향은 작음.
- 보정: 앞서 별도 확인(원본 CSV)에서 지연을 252건이라 적었으나 6-8 출력 기준 258건이 맞음. 그 외 노트북 밖 수치는 6-8과 일치해 이 항목으로 대체.

**우측절단(04 노트북의 D-17)과의 관계 — 6-10 추가**
- 04는 02_geo_distance에서 발견된 "8월 후반(8/15~31) 구매분의 late율이 우측절단·생존표본 편향으로 눌려 있다"는 점을 기록함(8/15~31 n=2,414, late 2.40%). 이를 제외해도 N=4 전체 late율은 4.40% → 4.61%(+0.21%p)로 판단기준(0.3~0.5%p) 미달이라 테스트 구간(D-03)은 유지됨. 그 판단은 **전체 late율**에 대한 것이고 distance와 late의 관계에 미치는 영향은 검토되지 않았음.
- 노트북 밖 근사(원본 CSV, 주문의 모든 셀러가 고객과 같은 주인 경우 vs 아닌 경우이며 6-10의 거리 구간과 정의가 다름): 같은 주 주문의 late율은 8/1~14에 14.9%(n=1,586)로 5~7월 3.1%(n=7,392)보다 크게 높고, 다른 주 주문은 4.2% vs 4.2%로 그대로. 8/15~31에는 다른 주 주문 late율이 0.5%(n=1,175, 정상 4.2%)로 낮고 다른 주 비중이 48.7%(5~7월 61.1%)로 줄어, 후반부에 멀리서 오는 지연 주문이 덜 관측된 흔적이 있음.
- 따라서 8월의 distance 역전은 ① 8월 초 가까운 거리(같은 주)에서의 실제 지연 급증과 ② 8월 후반 먼 거리 지연 주문의 과소 관측이 합쳐진 결과일 수 있음(추정). 이전 해석의 "구매 주별 late율 17.5% → 3.2%"에서 3~4주차의 낮은 값을 "후반이 개선됐다"로 읽지 않고, 일부는 절단 효과일 수 있다고 봄. 두 요인의 비중은 아래 6-10 결과 참고.
- **6-10 결과** (distance_km_filled 기준, 8/1~14 · 8/15~31 · 5~7월 순, 미매칭 477건은 주 평균으로 채워진 값 포함)
  - 거리 구간별 late율: 0~100km 20.0%(n=873) · 5.0%(727) · 3.3%(3,943) / 100~300 9.5 · 4.4 · 3.1 / 300~600 4.7 · 0.5 · 3.2 / 600~1000 5.6 · 0.5 · 3.0 / 1000+ 2.0 · 0.0 · 6.9
  - 1000km 이상 비중: 5~7월 15.9%, 8/1~14 14.9%, 8/15~31 8.3%(거의 절반). 0~100km 비중은 20.8% · 22.2% · 30.1%
  - distance 단독 AUC: 5~7월 0.568(양성 722) / 8/1~14 0.296(양성 335) / 8/15~31 0.248(양성 58)
  - **해석**: ① 8월 양성 393건 중 335건(85%)이 8/1~14에서 나오고, 이 구간에서 0~100km late율이 20.0%(5~7월 3.3%)로 뛰어 역전의 주된 원인은 8월 초 근거리 지연 급증으로 보임. ② 8/15~31은 300km 이상 late율이 0.0~0.5%이고 1000km 이상 비중이 절반으로 줄어 04의 우측절단·생존표본 편향 흔적과 일치하지만, 양성이 58건뿐이라 역전 전체에 대한 기여는 작음(AUC 0.248도 양성 58건 기준이라 약한 근거). ③ 8/1~14의 1000km 이상 late율(2.0%)이 5~7월(6.9%)보다 낮은 것은 양성이 약 12건(n=586)이라 불안정하고, 이 구간에서도 절단 영향이 있는지 이 출력만으로는 가를 수 없어 해석을 보류. 절단 여부는 미도착 비율로는 가를 수 없음(원본 CSV 확인, 노트북 밖 근사: 8월 구매의 도착 비율이 구간·지역별로 97~99%로 차이 없음). 절단은 미도착이 아니라 8월 말 주문이 테이블에 거의 없는 형태로 나타나므로, 6-8의 base 일별 주문 수 급감(8/22 185 → 8/29 11건)과 마지막 주 실제 소요일 2.1일로 판단. ④ 이전 해석 "8월 초 근거리 집중"은 유지하고, 3~4주차의 낮은 값을 '후반 개선'으로 읽지 않는 보정만 적용. 근거리 급증의 원인(운송·예상일 설정·기타)은 여전히 미확인.
  - 노트북 밖 근사(원본 CSV, 같은 주 여부 기준)의 8/1~14 같은 주 14.9%와 방향은 같고 정의가 달라 수치는 다름. 이 절에서는 6-10 값을 따름.


In [22]:
# 6-11. SP→SP 예상일·실제 소요일·여유 분포 (날짜 기준, 6-8과 같은 정의) — 8월 초 지연 급증이 예상일 쪽 변화인지 실제 배송 쪽 변화인지 확인
# late 라벨이 날짜 비교(CAST AS DATE)라서 여유도 날짜 단위로 계산한다. 예상일은 00:00이라 연속값 여유가 음수여도 예상일 당일 도착이면 지연이 아니다.
# 평균이 아니라 분포(중앙값·p90·p10)를 본다. late는 꼬리 지표다. 8/1~14는 8/1~7과 8/8~14로도 나눠 본다(일별 late율이 8/7까지 높다가 8/8에 급락).
# 구간은 주문 단위 부트스트랩(독립 가정, 실제보다 좁음).
d = master.copy()
P = d["purchase_ts"]
d["est"] = (d["order_estimated_delivery_date"] - P).dt.total_seconds() / 86400
d["act"] = (d["order_delivered_customer_date"] - P).dt.total_seconds() / 86400
d["slack_d"] = (d["order_estimated_delivery_date"].dt.normalize()
                - d["order_delivered_customer_date"].dt.normalize()).dt.days
assert ((d["slack_d"] < 0).astype(int) == d["late"]).all()      # 날짜 여유<0 ⇔ late
d = d[(d["customer_state"] == "SP") & (d["seller_state"] == "SP")]   # 6-8과 같은 SP→SP(대표 셀러 기준)

periods = {"6월": ("2018-06-01", "2018-07-01"), "7월": ("2018-07-01", "2018-08-01"),
           "8/1~7": ("2018-08-01", "2018-08-08"), "8/8~14": ("2018-08-08", "2018-08-15"),
           "8/1~14(합산)": ("2018-08-01", "2018-08-15"), "8/15~31(절단 영향, 참고)": ("2018-08-15", "2018-09-01")}
sub = {k: d[(d["purchase_ts"] >= a) & (d["purchase_ts"] < b)] for k, (a, b) in periods.items()}
print(pd.DataFrame([{"구간": k, "n": len(g), "예상일_중앙": g["est"].median(), "실제_중앙": g["act"].median(),
                     "실제_p90": g["act"].quantile(0.9), "여유(일)_중앙": g["slack_d"].median(),
                     "여유(일)_p10": g["slack_d"].quantile(0.1), "late율": g["late"].mean()}
                    for k, g in sub.items()]).round(2).to_string(index=False))

A = sub["7월"]
rng = np.random.default_rng(42)
stats = {"예상일 중앙값(일)": ("est", np.median), "실제 소요일 중앙값(일)": ("act", np.median),
         "실제 소요일 p90(일)": ("act", lambda v: np.percentile(v, 90)), "late율(%p)": ("late", lambda v: 100 * np.mean(v))}
for win in ["8/1~7", "8/8~14", "8/1~14(합산)"]:
    B = sub[win]
    print(f"\n[{win} − 7월 차이, 95% 구간(주문 단위 부트스트랩 2,000회)]")
    for name, (col, f) in stats.items():
        a, b = A[col].to_numpy(), B[col].to_numpy()
        diffs = [f(b[rng.integers(0, len(b), len(b))]) - f(a[rng.integers(0, len(a), len(a))]) for _ in range(2000)]
        print(f"  {name}: {f(b) - f(a):+.2f} [{np.percentile(diffs, 2.5):+.2f}, {np.percentile(diffs, 97.5):+.2f}]")


                구간    n  예상일_중앙  실제_중앙  실제_p90  여유(일)_중앙  여유(일)_p10  late율
                6월 2068   19.15   5.10   10.15      14.0        7.0   0.01
                7월 2040   13.06   5.08   10.19       7.0        2.0   0.04
             8/1~7  752    7.45   6.01   11.17       2.0       -4.0   0.22
            8/8~14  629    8.35   5.17   10.03       4.0        0.0   0.08
        8/1~14(합산) 1381    8.03   5.62   10.85       3.0       -2.0   0.15
8/15~31(절단 영향, 참고) 1100    8.11   4.41    8.11       4.0        0.0   0.04

[8/1~7 − 7월 차이, 95% 구간(주문 단위 부트스트랩 2,000회)]
  예상일 중앙값(일): -5.60 [-5.84, -4.94]
  실제 소요일 중앙값(일): +0.93 [+0.71, +1.14]
  실제 소요일 p90(일): +0.98 [+0.36, +2.05]
  late율(%p): +17.47 [+14.59, +20.63]

[8/8~14 − 7월 차이, 95% 구간(주문 단위 부트스트랩 2,000회)]
  예상일 중앙값(일): -4.71 [-5.04, -3.88]
  실제 소요일 중앙값(일): +0.09 [-0.11, +0.42]
  실제 소요일 p90(일): -0.16 [-1.08, +0.42]
  late율(%p): +3.88 [+1.68, +6.13]

[8/1~14(합산) − 7월 차이, 95% 구간(주문 단위 부트스트랩 2,000회)]
  예상일 중앙값(일): -5.03 [-5.62, -4.42]
  실제 소

### 6-11 해석 (SP→SP 대표 셀러 기준, 7월 vs 8월 초)

| | 7월 (n=2,040) | 8/1~7 (n=752) | 8/8~14 (n=629) | 8/1~14 합산 (n=1,381) |
|---|---|---|---|---|
| 예상소요일 중앙값 | 13.06일 | 7.45일 (−5.60 [−5.84, −4.94]) | 8.35일 (−4.71 [−5.04, −3.88]) | 8.03일 (−5.03 [−5.62, −4.42]) |
| 실제 소요일 중앙값 | 5.08일 | 6.01일 (+0.93 [+0.71, +1.14]) | 5.17일 (+0.09 [−0.11, +0.42]) | 5.62일 (+0.54 [+0.26, +0.86]) |
| 실제 소요일 p90 | 10.19일 | 11.17일 (+0.98 [+0.36, +2.05]) | 10.03일 (−0.16 [−1.08, +0.42]) | 10.85일 (+0.66 [−0.12, +1.14]) |
| 날짜 기준 여유 중앙값 / p10 | 7 / 2일 | 2 / −4일 | 4 / 0일 | 3 / −2일 |
| late율 | 4% | 22% (+17.47%p [+14.59, +20.63]) | 8% (+3.88%p [+1.68, +6.13]) | 15% (+11.28%p [+9.29, +13.46]) |

- **읽는 법**: late는 정의상 날짜 여유(예상일 − 도착일)가 음수인 주문이다. "여유 p10이 음수가 됐다"는 late 증가를 다시 말한 것이지 원인이 아니다. 의미 있는 것은 이동의 분해다.
- **분해**: 예상소요일은 두 주 모두 5일 안팎 짧아졌다(두 주 공통). 실제 소요일은 첫 주(8/1~7)에만 늘었고(중앙값 +0.93일, p90 +0.98일, 구간이 0 제외) 둘째 주(8/8~14)는 7월 수준이다(구간이 0 포함). late는 4% → 22% → 8%로 갈렸다. 8/1~14를 합치면 이 차이가 가려져 "실제 소요일은 소폭만 늘었다"로 읽히므로 합산만 보면 안 된다.
- **예상일 단축의 역할**: 실제 소요일이 7월 수준이었던 둘째 주(8/8~14)에도 late가 4% → 8%로 올랐으므로(+3.88%p [+1.68, +6.13]) 예상일 단축(−4.71일)이 지연에 취약하게 만든 조건으로 작용했을 가능성이 크다. 6월 → 7월은 실제 소요일이 같은데(중앙값 5.10 → 5.08일, p90 10.15 → 10.19일) 예상일이 약 6일 줄고 late가 1% → 4%로 올라 예상일 효과와 부합한다(6월은 8개월 중 late가 가장 낮은 달이라 상승폭 해석에는 주의). 5월 → 7월(6-8 표, 평균 기준: 예상일 16.4 → 12.9일, late 4.3% → 4.1%)만 같은 기간 실제 소요일도 줄어(7.3 → 6.1일) 효과의 유무를 가리지 못한다. 첫 주(8/1~7)의 22%는 여기에 실제 소요일 증가(+0.93일)가 겹친 것으로 본다. 기여 비중은 확정할 수 없다(6-12 반사실 근사 참고).
- **둘째 주(8/8~14)**: late(8%)는 첫 주(22%)보다 낮지만 7월(4%)보다 +3.88%p [+1.68, +6.13] 높고, 실제 소요일은 7월 수준이다. 실제 p90 차이(−0.16 [−1.08, +0.42])가 7월과 구분되지 않아 우측절단 신호는 없어 보이나 작은 절단은 배제하지 못한다. 뚜렷한 절단은 8/15~31(실제 p90 8.11일, late 4%)에서 나타난다(느린 주문이 빠짐 — 6-8의 일별 주문 수 급감과 부합). 이 구간은 비교 대상으로 쓰지 않음.
- **일별 추이(노트북 밖, 원본 CSV 근사)**: late율은 8/7(약 33%)까지 높다가 8/8(약 9.5%)에 급락한다. 노트북 출력에는 일별 값이 없다.
- **의미**: 8월 distance 역전은 distance 자체보다 라벨 기준(예상일)이 짧아진 시기의 SP 내 지연 급증과 함께 나타났다. late 라벨의 시간 일관성이 1단계의 한계로 추가된다.
- **한계**: 평균·분포 기반 비교이고 SP→SP 한정이며 7월과 8월 초만 비교했다. 구간은 주문 단위 독립 가정이라 실제보다 좁다. 8/1~7과 8/8~14의 경계는 일별 late율이 8/7까지 높다가 8/8에 급락하는 것을 본 뒤 정한 사후 탐색이며 경계 선택과 다중비교를 반영하지 않았다. 또 7월 기준(late 4.1%)에는 급증이 시작된 7/30~31(late 12.8%, 6-12)이 포함돼 있어 7/1~29(3.0%)보다 높게 잡혔다. 비교 기준을 7/1~29로 하면 차이는 오히려 커지므로 결론은 바뀌지 않는다. 예상일을 왜 줄였는지, 8/1~7에 실제로 느려진 원인(일별로 8/6~7이 특히 높음), 라벨 기준 변화가 지연 급증의 몇 %를 설명하는지는 이 자료로 확정할 수 없다.
- 이 해석을 반영하면 1단계 설명은 변수 중요도보다 월별·지역별 기술통계와 라벨 기준 변화 기록이 더 적절하다(강사님 질문 2, 3).


In [23]:
# 6-12. 예상일 단축 vs 실제 소요일 증가 — 반사실 분해(단순 근사)와 7월 기준선 점검 (SP→SP, 6-11과 같은 정의)
# 반사실은 "예상일을 7월 중앙값 수준으로 되돌렸을 때"(B)와 "실제 소요일을 7월 중앙값 수준으로 되돌렸을 때"(C)의 late율이다.
# 모든 주문의 예상일(또는 실제 소요일)을 같은 일수만큼 옮기는 단순 가정이고 SP→SP 한정이다. 원인 확정이 아니라 크기 가늠용이다.
# 사후 탐색(8/1~7·8/8~14 경계, 7/30 경계)이며 다중비교를 반영하지 않는다.
d = master.copy(); P = d["purchase_ts"]
d["est"] = (d["order_estimated_delivery_date"] - P).dt.total_seconds() / 86400
d["act"] = (d["order_delivered_customer_date"] - P).dt.total_seconds() / 86400
d["slack_d"] = (d["order_estimated_delivery_date"].dt.normalize() - d["order_delivered_customer_date"].dt.normalize()).dt.days
sp = d[(d["customer_state"] == "SP") & (d["seller_state"] == "SP")]
day = sp["purchase_ts"].dt.normalize()
jul = sp[(day >= "2018-07-01") & (day <= "2018-07-31")]
rows = []
for name, (a, b) in {"7/1~29": ("2018-07-01", "2018-07-29"), "7/30~31": ("2018-07-30", "2018-07-31")}.items():
    g = sp[(day >= a) & (day <= b)]
    rows.append({"구간": name, "n": len(g), "예상일_중앙": g["est"].median(), "실제_중앙": g["act"].median(), "late율": g["late"].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False)); print()

jm_est, jm_act = jul["est"].median(), jul["act"].median()
rows = []
for name, (a, b) in {"8/1~7": ("2018-08-01", "2018-08-07"), "8/8~14": ("2018-08-08", "2018-08-14")}.items():
    g = sp[(day >= a) & (day <= b)]
    sh_est = int(round(jm_est - g["est"].median())); sh_act = int(round(g["act"].median() - jm_act))
    rows.append({"구간": name, "n": len(g), "실제 late": g["late"].mean(),
                 f"B 예상일 +일수(7월 수준)": sh_est, "B late": ((g["slack_d"] + sh_est) < 0).mean(),
                 f"C 실제 −일수(7월 수준)": sh_act, "C late": ((g["slack_d"] + sh_act) < 0).mean(),
                 "7월 late": jul["late"].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))


     구간    n  예상일_중앙  실제_중앙  late율
 7/1~29 1813   13.27  5.126  0.030
7/30~31  227    8.05  3.987  0.128

    구간   n  실제 late  B 예상일 +일수(7월 수준)  B late  C 실제 −일수(7월 수준)  C late  7월 late
 8/1~7 752    0.215                 6   0.033                1   0.161    0.041
8/8~14 629    0.079                 5   0.022                0   0.079    0.041


### 6-12 해석 (예상일 단축 vs 실제 소요일 증가, SP→SP)

| 구간 | n | 예상일 중앙 | 실제 중앙 | late율 |
|---|---|---|---|---|
| 7/1~29 | 1,813 | 13.27일 | 5.13일 | 3.0% |
| 7/30~31 | 227 | 8.05일 | 3.99일 | 12.8% |

| 반사실 | 실제 late | B: 예상일을 7월 수준으로 | C: 실제 소요일을 7월 수준으로 |
|---|---|---|---|
| 8/1~7 | 약 22% | 3.3% (예상일 +6일) | 16.1% (실제 −1일) |
| 8/8~14 | 약 8% | 2.2% (예상일 +5일) | 7.9% (이동 0일) |

- **7/30~31**: 예상일은 이미 8.05일로 짧아졌는데(7/1~29는 13.27일) 실제 소요일은 3.99일로 7월(5.08일)보다 빨랐다. 실제 배송이 빨라졌는데 late가 12.8%이므로 예상일 쪽이 주요인이라는 가장 직접적인 근거다. 급증은 8월이 아니라 7/30경 시작됐고, 7월 기준 late(4.1%)에 이 이틀이 섞여 있다(7/1~29는 3.0%).
- **B와 C**: 예상일을 7월 수준으로 되돌리면 두 주 모두 late가 7월(4.1%)과 비슷하거나 낮은 2~3%대로 내려가고, 실제 소요일을 7월 수준으로 되돌려도 거의 내려가지 않는다(16.1%, 7.9%). 예상일 쪽 효과가 크게 나타난다. B와 C의 이동 일수가 달라(5~6일 vs 0~1일) 두 값을 "비중"으로 직접 비교하지는 않는다.
- **범위(노트북 밖 근사, 원본 CSV)**: 이동 일수를 +4~+6일로 바꾸면 8/1~7은 7.1/4.5/3.3%, 8/8~14는 2.7/2.2/1.9%이다(+3일이면 10.3%와 2.9%, +7일이면 2.3%와 1.0%). 예상일 분포를 순서를 유지하며 7월 분포로 바꾸는 분위 매핑은 구현에 따라 8/1~7 약 7~9%, 8/8~14 약 3%로, 같은 방향이다.
- **한계**: 모든 주문을 같은 일수만큼 옮기는 단순 가정이고 SP→SP 한정이다. 7월 주문의 예상일을 5일 줄이는 단순 적용(노트북 밖 근사)은 late 20.1%로 8/1~7(약 22%)과 비슷하지만 8/8~14(약 8%)는 따라가지 못해, 둘째 주에는 예상일 단축이 균일하지 않았다고 본다(예상일 분포의 하위·상위 꼬리는 약 3일, 중앙은 약 5일 짧아짐). 그래서 급증의 몇 %를 설명하는지는 확정하지 않는다. 8/1~7·8/8~14와 7/30 경계는 일별 late율을 본 뒤 정한 사후 탐색이며 다중비교를 반영하지 않았다.


## 7. 시간순 CV 튜닝 + 임계값 + 최종 테스트 평가 (D-20)

**왜 추가했나**: 계획서 5절은 1단계에 TimeSeriesSplit+하이퍼파라미터 튜닝을 요구했는데, 2~5절의 RF(depth 8, min_samples_leaf 50)와 로지스틱(기본값)은 근거 없는 임시 고정값이었다. 계획서를 이행하는 작업이며 계획 변경이 아니다.

**규칙은 7절 실행 전에 `DCV_cv_tuning.md`에 정리했다**(작성자 기록이며 커밋 이력이 없어 시점은 증명하지 못함. decisions.md D-20으로의 이전은 실행 후). 이 절의 요지:
- 튜닝·임계값 선택은 **학습 구간(purchase_ts < 2018-05-01)만** 사용. 7-1~7-4(그리드 셀)는 테스트 마스크를 참조하지 않는다.
- 선택 지표: 폴드별 PR-AUC ÷ 폴드 기저율의 평균(폴드마다 late율이 달라 PR-AUC를 그대로 평균하지 않음). 동률(차이 < 0.005)이면 단순한 설정.
- 테스트는 7-3, 7-5에서 선택된 설정으로 **한 번만** 평가. 결과를 보고 설정을 다시 고르지 않는다. (2~6절에서 테스트 구간 진단은 이미 봤으므로, 정확한 의미는 '설정 선택에 테스트를 쓰지 않았다'이다.)
- 한계: ① 첫 폴드는 학습 구간이 2017 초반뿐이라 양성이 적음 ② `distance_km_filled`의 주 평균과 `category_grouped` 상위 22개는 폴드와 무관하게 학습 구간(전체/전체 기간)으로 계산돼 있음(영향 작을 것으로 보이나 미계산) ③ gap은 행 수 기준이라 기간으로는 2017년에 더 김

In [24]:
# 7-1. CV 설정 — 학습 구간만 사용 (테스트 마스크 미참조)
import itertools, time
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import precision_recall_curve

N_SPLITS = 5
GAP = 5000          # D-20: 학습/검증 사이 간격(행). 아직 배송 전인 주문의 라벨이 학습에 섞이는 것을 줄이는 목적
N_EST = 300         # RF 트리 수 (기존과 동일)

tr1 = master[train_mask].sort_values(["purchase_ts", "order_id"]).reset_index(drop=True)
tr2 = tr1[tr1["low"].notna()].reset_index(drop=True)
assert tr1["purchase_ts"].max() < pd.Timestamp(TRAIN_END)

def make_splits(n):
    return list(TimeSeriesSplit(n_splits=N_SPLITS, gap=GAP).split(np.arange(n)))

def describe_splits(df, ycol, splits):
    rows = []
    for k, (a, b) in enumerate(splits, 1):
        rows.append({"fold": k, "n_train": len(a), "n_valid": len(b),
                     "valid_기간": f"{df['purchase_ts'].iloc[b[0]]:%Y-%m-%d}~{df['purchase_ts'].iloc[b[-1]]:%Y-%m-%d}",
                     "valid_양성": int(df[ycol].iloc[b].astype(int).sum()),
                     "valid_기저율": round(df[ycol].iloc[b].astype(int).mean(), 4)})
    return pd.DataFrame(rows)

splits1, splits2 = make_splits(len(tr1)), make_splits(len(tr2))
print("[1단계 late] 학습 구간", len(tr1), "건"); print(describe_splits(tr1, "late", splits1).to_string(index=False))
print("\n[2단계 low] 학습 구간", len(tr2), "건"); print(describe_splits(tr2, "low", splits2).to_string(index=False))

[1단계 late] 학습 구간 70852 건
 fold  n_train  n_valid              valid_기간  valid_양성  valid_기저율
    1     6812    11808 2017-06-09~2017-09-12       369     0.0312
    2    18620    11808 2017-09-12~2017-11-24       769     0.0651
    3    30428    11808 2017-11-24~2018-01-17      1007     0.0853
    4    42236    11808 2018-01-17~2018-03-09      1701     0.1441
    5    54044    11808 2018-03-09~2018-04-30      1104     0.0935

[2단계 low] 학습 구간 70319 건
 fold  n_train  n_valid              valid_기간  valid_양성  valid_기저율
    1     6724    11719 2017-06-09~2017-09-12      1166     0.0995
    2    18443    11719 2017-09-12~2017-11-24      1439     0.1228
    3    30162    11719 2017-11-24~2018-01-17      1776     0.1515
    4    41881    11719 2018-01-17~2018-03-09      2261     0.1929
    5    53600    11719 2018-03-09~2018-04-30      1748     0.1492


In [25]:
# 7-2. 공통 함수 + 1단계 그리드 (로지스틱 C 4개, RF depth×leaf 6개; class_weight는 balanced 고정 — 계획서 3절)
def cv_run(df, ycol, num, bin_, cat, make_clf, splits):
    X = df[num + bin_ + cat]; y = df[ycol].astype(int).to_numpy()
    ratios, oof = [], np.full(len(df), np.nan)
    for a, b in splits:
        if len(np.unique(y[a])) < 2 or len(np.unique(y[b])) < 2:
            ratios.append(np.nan); continue
        pipe = Pipeline([("pre", make_preprocessor(num, bin_, cat)), ("clf", make_clf())])
        pipe.fit(X.iloc[a], y[a])
        p = pipe.predict_proba(X.iloc[b])[:, 1]
        ratios.append(average_precision_score(y[b], p) / y[b].mean())
        oof[b] = p
    return np.array(ratios), oof

def run_grid(df, ycol, num, bin_, cat, grid, splits):
    rows, oofs = [], {}
    for name, make_clf, simp in grid:
        t0 = time.time()
        r, oof = cv_run(df, ycol, num, bin_, cat, make_clf, splits)
        rows.append({"config": name, "mean_ratio": np.nanmean(r), "min_ratio": np.nanmin(r),
                     **{f"f{k+1}": v for k, v in enumerate(r)}, "simp": simp, "sec": round(time.time() - t0)})
        oofs[name] = oof
    return pd.DataFrame(rows), oofs

def pick(tbl, tol=0.005):
    """D-20: 최고 mean_ratio와 tol 이내인 설정 중 가장 단순한(simp 작은) 설정"""
    cand = tbl[tbl["mean_ratio"] >= tbl["mean_ratio"].max() - tol]
    return cand.sort_values("simp").iloc[0]["config"]

def logit_f(C, cw): return lambda: LogisticRegression(C=C, class_weight=cw, max_iter=1000, random_state=42)
def rf_f(d, l, cw): return lambda: RandomForestClassifier(n_estimators=N_EST, class_weight=cw, max_depth=d,
                                                            min_samples_leaf=l, random_state=42, n_jobs=-1)
CW_RANK = {None: 0, "balanced": 1}   # 불균형이 완만하면 미적용 쪽이 단순 (계획서 5절 2단계)

grid1_logit = [(f"logit C={C}", logit_f(C, "balanced"), (C,)) for C in [0.01, 0.1, 1, 10]]
grid1_rf = [(f"rf depth={d} leaf={l}", rf_f(d, l, "balanced"), (d, -l)) for d in [4, 8, 12] for l in [50, 200]]

num1, bin1, cat1 = STAGE1_NUM, STAGE1_BIN, STAGE1_CAT
tbl1_logit, oof1_logit = run_grid(tr1, "late", num1, bin1, cat1, grid1_logit, splits1)
tbl1_rf, oof1_rf = run_grid(tr1, "late", num1, bin1, cat1, grid1_rf, splits1)
print("[1단계 로지스틱] 폴드별 PR-AUC/기저율 (1.0 = 기저율 수준)"); print(tbl1_logit.drop(columns="simp").round(3).to_string(index=False))
print("\n[1단계 RF]"); print(tbl1_rf.drop(columns="simp").round(3).to_string(index=False))

[1단계 로지스틱] 폴드별 PR-AUC/기저율 (1.0 = 기저율 수준)
      config  mean_ratio  min_ratio    f1    f2    f3    f4    f5  sec
logit C=0.01       1.350      1.192 1.598 1.302 1.192 1.290 1.366    1
 logit C=0.1       1.334      1.189 1.573 1.264 1.189 1.282 1.360    1
   logit C=1       1.324      1.188 1.553 1.237 1.188 1.282 1.357    1
  logit C=10       1.322      1.188 1.545 1.235 1.188 1.282 1.358    1

[1단계 RF]
              config  mean_ratio  min_ratio    f1    f2    f3    f4    f5  sec
  rf depth=4 leaf=50       1.421      1.114 1.699 1.285 1.114 1.363 1.647    7
 rf depth=4 leaf=200       1.404      1.090 1.491 1.356 1.090 1.365 1.715    7
  rf depth=8 leaf=50       1.382      1.109 1.544 1.258 1.109 1.351 1.649    8
 rf depth=8 leaf=200       1.408      1.082 1.478 1.382 1.082 1.391 1.709    8
 rf depth=12 leaf=50       1.367      1.105 1.513 1.268 1.105 1.339 1.608    9
rf depth=12 leaf=200       1.382      1.058 1.478 1.356 1.058 1.374 1.643    8


In [26]:
# 7-3. 1단계 선택 + OOF 임계값(F1 최대, D-20) + 테스트 한 번만 평가
def best_f1_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    f1 = 2 * pr[:-1] * rc[:-1] / np.clip(pr[:-1] + rc[:-1], 1e-12, None)
    i = f1.argmax()
    return th[i], f1[i]

def lift_top(y, p, frac=0.10):
    k = max(int(len(y) * frac), 1); idx = np.argsort(-p)[:k]
    return y[idx].mean() / y.mean()

def final_eval(name, make_clf, tr, ycol, num, bin_, cat, te, thr):
    pipe = Pipeline([("pre", make_preprocessor(num, bin_, cat)), ("clf", make_clf())])
    pipe.fit(tr[num + bin_ + cat], tr[ycol].astype(int))
    y = te[ycol].astype(int).to_numpy(); p = pipe.predict_proba(te[num + bin_ + cat])[:, 1]
    pred = (p >= thr).astype(int)
    out = {"name": name, "base_rate": y.mean(), "pr_auc": average_precision_score(y, p),
           "roc_auc": roc_auc_score(y, p), "lift_top10": lift_top(y, p), "thr": thr}
    print(f"--- {name} (threshold={thr:.3f}) ---")
    print(f"기저율 {out['base_rate']:.4f} / PR-AUC {out['pr_auc']:.4f} ({out['pr_auc']/out['base_rate']:.2f}배) / ROC-AUC {out['roc_auc']:.4f} / 상위 10% 리프트 {out['lift_top10']:.2f}")
    print(confusion_matrix(y, pred)); print(classification_report(y, pred, digits=3, zero_division=0))
    return out

te1 = master[test_mask]       # 이 셀에서 처음 테스트를 사용함 — 이 셀은 한 번만 실행
sel1_logit, sel1_rf = pick(tbl1_logit), pick(tbl1_rf)
print("선택:", sel1_logit, "/", sel1_rf)
cv_results = []
for sel, tbl, oofs, grid in [(sel1_logit, tbl1_logit, oof1_logit, grid1_logit), (sel1_rf, tbl1_rf, oof1_rf, grid1_rf)]:
    oof = oofs[sel]; ok = ~np.isnan(oof)
    thr, f1 = best_f1_threshold(tr1.loc[ok, "late"].astype(int).to_numpy(), oof[ok])
    print(f"{sel}: OOF F1 최대 임계값 {thr:.3f} (OOF F1 {f1:.3f})")
    make = [g[1] for g in grid if g[0] == sel][0]
    cv_results.append(final_eval(f"1단계 {sel} [CV선택]", make, tr1, "late", num1, bin1, cat1, te1, thr))

선택: logit C=0.01 / rf depth=4 leaf=50
logit C=0.01: OOF F1 최대 임계값 0.519 (OOF F1 0.168)
--- 1단계 logit C=0.01 [CV선택] (threshold=0.519) ---
기저율 0.0440 / PR-AUC 0.0523 (1.19배) / ROC-AUC 0.5046 / 상위 10% 리프트 1.44
[[11730 12507]
 [  554   561]]
              precision    recall  f1-score   support

           0      0.955     0.484     0.642     24237
           1      0.043     0.503     0.079      1115

    accuracy                          0.485     25352
   macro avg      0.499     0.494     0.361     25352
weighted avg      0.915     0.485     0.618     25352

rf depth=4 leaf=50: OOF F1 최대 임계값 0.513 (OOF F1 0.181)
--- 1단계 rf depth=4 leaf=50 [CV선택] (threshold=0.513) ---
기저율 0.0440 / PR-AUC 0.0457 (1.04배) / ROC-AUC 0.4768 / 상위 10% 리프트 1.33
[[ 9981 14256]
 [  540   575]]
              precision    recall  f1-score   support

           0      0.949     0.412     0.574     24237
           1      0.039     0.516     0.072      1115

    accuracy                          0.416     25352
   ma

In [27]:
# 7-4. 2단계 그리드 (class_weight None/balanced도 함께 — 계획서 5절 2단계: 불균형이 완만하면 미적용 우선)
num2 = STAGE1_NUM + STAGE2_NUM
grid2_logit = [(f"logit C={C} cw={cw}", logit_f(C, cw), (CW_RANK[cw], C)) for C in [0.01, 0.1, 1, 10] for cw in [None, "balanced"]]
grid2_rf = [(f"rf depth={d} leaf={l} cw={cw}", rf_f(d, l, cw), (CW_RANK[cw], d, -l))
            for d in [4, 8, 12] for l in [50, 200] for cw in [None, "balanced"]]
tbl2_logit, oof2_logit = run_grid(tr2, "low", num2, bin1, cat1, grid2_logit, splits2)
tbl2_rf, oof2_rf = run_grid(tr2, "low", num2, bin1, cat1, grid2_rf, splits2)
print("[2단계 로지스틱]"); print(tbl2_logit.drop(columns="simp").round(3).to_string(index=False))
print("\n[2단계 RF]"); print(tbl2_rf.drop(columns="simp").round(3).to_string(index=False))

[2단계 로지스틱]
                  config  mean_ratio  min_ratio    f1    f2    f3    f4    f5  sec
    logit C=0.01 cw=None       1.377      1.254 1.254 1.505 1.406 1.346 1.376    1
logit C=0.01 cw=balanced       1.381      1.247 1.247 1.501 1.407 1.360 1.388    1
     logit C=0.1 cw=None       1.357      1.228 1.228 1.472 1.384 1.332 1.371    1
 logit C=0.1 cw=balanced       1.366      1.220 1.220 1.481 1.392 1.354 1.385    1
       logit C=1 cw=None       1.352      1.216 1.216 1.468 1.376 1.330 1.368    1
   logit C=1 cw=balanced       1.365      1.213 1.213 1.480 1.391 1.354 1.385    1
      logit C=10 cw=None       1.351      1.213 1.213 1.465 1.376 1.331 1.368    1
  logit C=10 cw=balanced       1.364      1.211 1.211 1.480 1.391 1.354 1.385    1

[2단계 RF]
                          config  mean_ratio  min_ratio    f1    f2    f3    f4    f5  sec
      rf depth=4 leaf=50 cw=None       1.397      1.257 1.257 1.466 1.408 1.411 1.445    7
  rf depth=4 leaf=50 cw=balanced       1.393      

In [28]:
# 7-5. 2단계 선택 + OOF 임계값 + 테스트 한 번만 평가
te2 = te1[te1["low"].notna()]
sel2_logit, sel2_rf = pick(tbl2_logit), pick(tbl2_rf)
print("선택:", sel2_logit, "/", sel2_rf)
for sel, oofs, grid in [(sel2_logit, oof2_logit, grid2_logit), (sel2_rf, oof2_rf, grid2_rf)]:
    oof = oofs[sel]; ok = ~np.isnan(oof)
    thr, f1 = best_f1_threshold(tr2.loc[ok, "low"].astype(int).to_numpy(), oof[ok])
    print(f"{sel}: OOF F1 최대 임계값 {thr:.3f} (OOF F1 {f1:.3f})")
    make = [g[1] for g in grid if g[0] == sel][0]
    cv_results.append(final_eval(f"2단계 {sel} [CV선택]", make, tr2, "low", num2, bin1, cat1, te2, thr))

선택: logit C=0.01 cw=None / rf depth=8 leaf=50 cw=None
logit C=0.01 cw=None: OOF F1 최대 임계값 0.111 (OOF F1 0.264)
--- 2단계 logit C=0.01 cw=None [CV선택] (threshold=0.111) ---
기저율 0.1000 / PR-AUC 0.1353 (1.35배) / ROC-AUC 0.5727 / 상위 10% 리프트 1.59
[[ 2234 20483]
 [  183  2342]]
              precision    recall  f1-score   support

           0      0.924     0.098     0.178     22717
           1      0.103     0.928     0.185      2525

    accuracy                          0.181     25242
   macro avg      0.513     0.513     0.181     25242
weighted avg      0.842     0.181     0.178     25242

rf depth=8 leaf=50 cw=None: OOF F1 최대 임계값 0.114 (OOF F1 0.268)
--- 2단계 rf depth=8 leaf=50 cw=None [CV선택] (threshold=0.114) ---
기저율 0.1000 / PR-AUC 0.1350 (1.35배) / ROC-AUC 0.5708 / 상위 10% 리프트 1.60
[[ 4974 17743]
 [  443  2082]]
              precision    recall  f1-score   support

           0      0.918     0.219     0.354     22717
           1      0.105     0.825     0.186      2525

    accurac

In [29]:
# 7-6. 요약 — 임시 고정값(2~5절) vs CV 선택값
cv_df = pd.DataFrame(cv_results)
cv_df["PR-AUC/기저율"] = cv_df["pr_auc"] / cv_df["base_rate"]
print(cv_df[["name", "base_rate", "pr_auc", "PR-AUC/기저율", "roc_auc", "lift_top10", "thr"]].round(4).to_string(index=False))
old = pd.DataFrame(results)[["name", "base_rate", "pr_auc", "roc_auc"]].head(5).copy()
old["PR-AUC/기저율"] = old["pr_auc"] / old["base_rate"]
print("\n[참고] 임시 고정값 결과"); print(old.round(4).to_string(index=False))

                                 name  base_rate  pr_auc  PR-AUC/기저율  roc_auc  lift_top10    thr
              1단계 logit C=0.01 [CV선택]      0.044  0.0523      1.1892   0.5046      1.4441 0.5188
        1단계 rf depth=4 leaf=50 [CV선택]      0.044  0.0457      1.0385   0.4768      1.3275 0.5135
      2단계 logit C=0.01 cw=None [CV선택]      0.100  0.1353      1.3521   0.5727      1.5882 0.1108
2단계 rf depth=8 leaf=50 cw=None [CV선택]      0.100  0.1350      1.3494   0.5708      1.5962 0.1140

[참고] 임시 고정값 결과
                            name  base_rate  pr_auc  roc_auc  PR-AUC/기저율
                        1단계 로지스틱      0.044  0.0524   0.5062      1.1913
                          1단계 RF      0.044  0.0519   0.4812      1.1791
2단계 로지스틱 (class_weight=balanced)      0.100  0.1362   0.5755      1.3618
    2단계 로지스틱 (class_weight=None)      0.100  0.1355   0.5733      1.3549
                          2단계 RF      0.100  0.1338   0.5706      1.3372


### 7절 해석 (실행 결과 반영)

- **폴드 구성(양성 수)**: 1단계 폴드별 검증 양성 369 / 769 / 1,007 / 1,701 / 1,104건(검증 기저율 3.1% ~ 14.4%), 2단계 1,166 / 1,439 / 1,776 / 2,261 / 1,748건(9.9% ~ 19.3%). 폴드마다 기저율이 달라 PR-AUC÷기저율로 비교함.
- **1단계 선택과 CV 점수**: 로지스틱 C=0.01(1.350배), RF depth 4·leaf 50(1.421배). 둘 다 그리드 가장자리라 더 강한 정규화가 나을 수 있으나, 테스트를 본 뒤 그리드를 넓히면 D-20 위반이므로 확장하지 않음(한계로 기록).
- **2단계 선택과 class_weight 결정**: 로지스틱 C=0.01 cw=None(1.377배), RF depth 8·leaf 50 cw=None(1.421배). class_weight는 **미적용**으로 결정. 근거: 로지스틱(C=0.01)은 balanced가 0.004 높아 D-20 동률 규칙(0.005 이내면 미적용 우선, 계획서 5절 "불균형이 완만하면 미적용")으로 미적용이 되고, RF(depth 8·leaf 50)는 미적용이 0.008 높아(1.421 vs 1.413) 규칙과 무관하게 최고값으로 선택됨.
- **테스트 결과(임시 고정값 대비)**: 로지스틱과 2단계는 변화 없음, 1단계 RF는 하락. 분모는 1단계 25,352건, 2단계 25,242건.
  - 1단계 PR-AUC: 로지스틱 0.0523(임시 0.0524, 기저율 0.044의 1.19배), RF 0.0457(임시 0.0519, 1.04배, ROC-AUC 0.477). RF의 0.006 감소는 신뢰구간을 구하지 않아 노이즈일 수 있음.
  - 2단계 PR-AUC: 로지스틱 0.1353(임시 0.1355), RF 0.1350(임시 0.1338). 둘 다 기저율 0.100의 1.35배.
  - 결론: 튜닝은 로지스틱과 2단계 성능을 바꾸지 못했고 1단계 RF는 오히려 낮아졌음(PR-AUC 0.0519 → 0.0457, 1.18배 → 1.04배, 신뢰구간 미계산이라 노이즈 가능). 신호가 약하다는 진단을 더 강하게 뒷받침함.
- **CV와 테스트의 차이**: 1단계는 선택 모델의 CV가 1.35~1.42배(단일 변수 1.47배)인데 테스트 1.04~1.19배로 어긋나고, 2단계는 선택 설정의 CV 1.38~1.42배, 테스트 1.35배로 거의 일치. 7-9에서 8월을 제외하면 1단계 테스트가 1.32~1.57배로 올라 CV 수준에 가까워지나, 순위는 여전히 CV와 다르다(CV는 단일 변수 > RF > 로지스틱, 8월 제외 테스트는 로지스틱 > 단일 변수 > RF). 수준 차이의 상당 부분은 8월(0.67~0.77배)로 설명되지만 로지스틱과 단일 변수의 순위 차이는 8월을 빼도 남고, CV가 시기 효과로 부풀었는지는 이 비교로는 확인되지 않음.
- **임계값 기준(OOF F1 최대)의 현상**: 신호가 약해 거의 전부를 경보로 두는 쪽으로 잡힘. 테스트에서 양성(경보) 표시 비율은 1단계 로지스틱 약 51%, RF 약 58%, 2단계 로지스틱 약 90%, RF 약 79%이고 정밀도는 기저율과 같은 수준(1단계 4.3%/3.9% vs 4.4%, 2단계 10.3%/10.5% vs 10.0%). 혼동행렬은 의미가 약하므로 보고는 PR-AUC와 상위 10% 리프트(1단계 1.44/1.33, 2단계 1.59/1.60)를 중심으로 하되, 리프트도 신뢰구간 미계산이고 1단계 테스트 양성의 약 75%가 5월(443건)·8월(393건)에 몰려 있어 독립 가정이 맞지 않으므로 강하게 인용하지 않음.
- **한계**: ① 첫 폴드 학습이 2017 초반뿐 ② `distance_km_filled`의 주 평균은 학습 구간 기준이고 `category_grouped` 상위 22개는 전체 기간(테스트 포함) 주문 수 기준이라 둘 다 폴드와 무관함(라벨을 쓰지 않아 누수는 아니나 영향 미계산) ③ gap이 행 수 기준 ④ 선택값이 그리드 가장자리 ⑤ 테스트 지표의 신뢰구간 미계산.


### 7-7. 베이스라인 기준선 (D-21)

"베이스라인 대비 개선폭"의 기준을 두 단계로 정의했다(모델 테스트 결과(7-3, 7-5)를 본 뒤에 정의함. R1 변수·방향은 학습 CV만으로 정하고 판정 규칙은 모델 결과에 의존하지 않음. decisions.md D-21로의 기록은 실행 후).
- **R0 기저율**: 상수 예측. PR-AUC = 테스트 기저율, 배수 1.0.
- **R1 단일 변수 규칙**: 후보 수치형 변수 하나를 점수로 쓴다. 변수와 방향(부호)은 학습 구간에서만 정한다 — D-20과 같은 5폴드에서 폴드 학습분의 스피어만 부호로 방향을 정하고, 검증분의 PR-AUC÷기저율 평균이 가장 높은 변수를 고른다. 테스트는 선택 후 한 번만 평가.
- 판정 규칙(7-7 코드에 명시): 주문 단위 부트스트랩 1,000회의 95% 구간이 ① 모델 배수가 1.0을 포함하면 "기저율과 구분되지 않음", ② 모델−R1 차이가 0을 포함하면 "단일 변수 대비 개선 확인 안 됨".
- 한계: 주문 단위 독립을 가정해 구간이 실제보다 좁다(1단계 테스트 양성의 약 75%가 5월·8월에 몰려 있음). 월 단위 블록 부트스트랩은 테스트가 4개월뿐이라 불가.

In [30]:
# 7-7. R1 단일 변수 규칙 + 부트스트랩 (테스트는 7-3/7-5의 선택 설정으로 다시 적합해 사용 — 설정 변경 없음)
from scipy.stats import spearmanr
N_BOOT = 1000

def _sign(x, y):
    r = spearmanr(x, y)[0]
    return 1.0 if (r is None or np.isnan(r) or r >= 0) else -1.0

def single_var_cv(df, ycol, cols, splits):
    rows = []
    for c in cols:
        r = []
        for a, b in splits:
            ya = df[ycol].iloc[a].astype(int).to_numpy(); yb = df[ycol].iloc[b].astype(int).to_numpy()
            s = _sign(df[c].iloc[a].to_numpy(), ya)
            r.append(average_precision_score(yb, s * df[c].iloc[b].to_numpy()) / yb.mean())
        rows.append({"var": c, "mean_ratio": np.mean(r), "min_ratio": np.min(r), **{f"f{k+1}": v for k, v in enumerate(r)}})
    return pd.DataFrame(rows).sort_values("mean_ratio", ascending=False).reset_index(drop=True)

def fit_prob(make_clf, tr, ycol, num, bin_, cat, te):
    pipe = Pipeline([("pre", make_preprocessor(num, bin_, cat)), ("clf", make_clf())])
    pipe.fit(tr[num + bin_ + cat], tr[ycol].astype(int))
    return pipe.predict_proba(te[num + bin_ + cat])[:, 1]

def boot_ratios(y, probs, n_boot=N_BOOT, seed=42):
    rng = np.random.default_rng(seed); n = len(y)
    out = {k: [] for k in probs}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n); yb = y[idx]
        if yb.sum() == 0: continue
        base = yb.mean()
        for k, p in probs.items(): out[k].append(average_precision_score(yb, p[idx]) / base)
    return {k: np.array(v) for k, v in out.items()}

def stage_report(label, tr, te, ycol, cols, splits, models):
    cv = single_var_cv(tr, ycol, cols, splits)
    print(f"[{label}] 후보 단일 변수 — 학습 구간 5폴드 PR-AUC/기저율"); print(cv.round(3).to_string(index=False))
    var = cv.loc[0, "var"]; s = _sign(tr[var].to_numpy(), tr[ycol].astype(int).to_numpy())
    y = te[ycol].astype(int).to_numpy()
    r1 = f"R1 {var}({'+' if s > 0 else '-'})"
    probs = {r1: s * te[var].to_numpy()}
    for name, mk, num, bin_, cat in models: probs[name] = fit_prob(mk, tr, ycol, num, bin_, cat, te)
    bs = boot_ratios(y, probs); base = y.mean(); rows = []
    for k, p in probs.items():
        pt = average_precision_score(y, p) / base
        lo, hi = np.percentile(bs[k], [2.5, 97.5])
        row = {"name": k, "PR-AUC/기저율": pt, "95%하한": lo, "95%상한": hi}
        if k != r1:
            d = bs[k] - bs[r1]; row.update({"Δvs_R1": pt - average_precision_score(y, probs[r1]) / base,
                                            "Δ하한": np.percentile(d, 2.5), "Δ상한": np.percentile(d, 97.5)})
        rows.append(row)
    print(f"\n[{label}] 테스트 (기저율 {base:.4f}, n={len(y)}, 부트스트랩 {N_BOOT}회, 주문 단위 독립 가정)")
    print(pd.DataFrame(rows).round(3).to_string(index=False))

mk1l = [g[1] for g in grid1_logit if g[0] == sel1_logit][0]; mk1r = [g[1] for g in grid1_rf if g[0] == sel1_rf][0]
mk2l = [g[1] for g in grid2_logit if g[0] == sel2_logit][0]; mk2r = [g[1] for g in grid2_rf if g[0] == sel2_rf][0]
stage_report("1단계 late", tr1, te1, "late", STAGE1_NUM, splits1,
             [(f"모델 {sel1_logit}", mk1l, num1, bin1, cat1), (f"모델 {sel1_rf}", mk1r, num1, bin1, cat1)])
print()
stage_report("2단계 low", tr2, te2, "low", STAGE1_NUM + STAGE2_NUM, splits2,
             [(f"모델 {sel2_logit}", mk2l, num2, bin1, cat1), (f"모델 {sel2_rf}", mk2r, num2, bin1, cat1)])

[1단계 late] 후보 단일 변수 — 학습 구간 5폴드 PR-AUC/기저율
               var  mean_ratio  min_ratio    f1    f2    f3    f4    f5
distance_km_filled       1.474      1.232 1.566 1.381 1.232 1.298 1.893
  seller_avg_delay       1.044      0.840 1.252 0.840 0.997 1.173 0.961

[1단계 late] 테스트 (기저율 0.0440, n=25352, 부트스트랩 1000회, 주문 단위 독립 가정)
                    name  PR-AUC/기저율  95%하한  95%상한  Δvs_R1    Δ하한   Δ상한
R1 distance_km_filled(+)       1.054  0.988  1.149     NaN    NaN   NaN
         모델 logit C=0.01       1.189  1.101  1.316   0.135  0.054 0.210
   모델 rf depth=4 leaf=50       1.038  0.978  1.124  -0.016 -0.090 0.046

[2단계 low] 후보 단일 변수 — 학습 구간 5폴드 PR-AUC/기저율
               var  mean_ratio  min_ratio    f1    f2    f3    f4    f5
       freight_sum       1.404      1.311 1.311 1.469 1.433 1.386 1.422
         price_sum       1.182      1.146 1.200 1.161 1.226 1.176 1.146
distance_km_filled       1.178      1.116 1.116 1.175 1.119 1.153 1.328
  installments_max       1.100      1.059 1.059 1.089 1.14

In [31]:
# 7-8. 모델 입력 변수만으로 VIF 재계산 (04 7-1의 후속)
# 판정 기준(코드에 명시, 시점 미증명): VIF >= 5 주의, >= 10 심각. 학습 구간(train_mask)만 사용.
# statsmodels 없이 상관행렬의 역행렬 대각으로 계산 (VIF_j = [R^-1]_jj, R = 상관행렬).

def vif_table(df, cols):
    X = df[cols].astype(float)
    X = X.loc[:, X.std() > 0]                      # 상수 열 제외
    R = np.corrcoef(X.to_numpy(), rowvar=False)
    v = np.diag(np.linalg.inv(R))
    return pd.DataFrame({"var": X.columns, "VIF": v}).sort_values("VIF", ascending=False).reset_index(drop=True)

trn = master[train_mask]

s1_cols = STAGE1_NUM + STAGE1_BIN                     # 수치 + 결측/폴백 더미
s2_cols = STAGE1_NUM + STAGE2_NUM + STAGE1_BIN

print(f"[1단계 변수] n={len(trn)}"); print(vif_table(trn, s1_cols).round(2).to_string(index=False))
print(f"\n[2단계 변수] n={trn['low'].notna().sum()}")
print(vif_table(trn[trn['low'].notna()], s2_cols).round(2).to_string(index=False))

# 카테고리 더미를 넣은 버전 (기준 범주는 가장 큰 범주를 빼서 고름)
dm = pd.get_dummies(trn["category_grouped"], prefix="cat").astype(float)
ref = dm.sum().idxmax()
dm = dm.drop(columns=ref)
print(f"\n[카테고리 더미 포함, 기준 범주={ref}]")
print(vif_table(pd.concat([trn[s2_cols], dm], axis=1), s2_cols + list(dm.columns)).head(10).round(2).to_string(index=False))

[1단계 변수] n=70852
                         var  VIF
seller_avg_delay_is_fallback 1.01
            seller_avg_delay 1.01
          distance_km_filled 1.00
                dist_missing 1.00

[2단계 변수] n=70319
                         var  VIF
                 freight_sum 1.35
                   price_sum 1.31
          distance_km_filled 1.12
            installments_max 1.12
seller_avg_delay_is_fallback 1.02
            seller_avg_delay 1.01
                dist_missing 1.00

[카테고리 더미 포함, 기준 범주=cat_bed_bath_table]
                      var  VIF
                   cat_기타 1.80
       cat_sports_leisure 1.73
        cat_health_beauty 1.70
cat_computers_accessories 1.64
      cat_furniture_decor 1.59
           cat_housewares 1.48
        cat_watches_gifts 1.47
              freight_sum 1.45
            cat_telephony 1.42
                 cat_toys 1.40


### 7-7 해석 (실행 결과 반영)

분모: 1단계 테스트 25,352건(기저율 0.0440), 2단계 25,242건(기저율 0.1000). 구간은 주문 단위 부트스트랩 1,000회 95%이며 독립 가정으로 실제보다 좁다.

- **1단계 R1**: `distance_km_filled`(+)가 선택됨(학습 5폴드 평균 1.474배, 폴드 1.23~1.89). 테스트에서는 1.054배 [0.988, 1.149]로 구간이 1.0을 포함 → 기저율과 구분되지 않음. 학습 구간 CV에서는 R1(1.474)이 튜닝된 모델(로지스틱 1.350, RF 1.421)보다 높았으나 테스트에서는 이 순위가 바뀜. distance 관계가 8월에 약해진 6절 진단과 같은 방향이지만, 이 비교 자체가 원인을 증명하지는 않음.
- **1단계 모델**: 로지스틱 1.189배 [1.101, 1.316], R1 대비 +0.135 [0.054, 0.210] → 기저율·R1과 모두 구분됨(독립 가정 하). RF 1.038배 [0.978, 1.124], R1 대비 −0.016 [−0.090, 0.046] → 둘 다 구분되지 않음.
- **2단계 R1**: `freight_sum`(+)이 선택됨(학습 평균 1.404배, 폴드 1.31~1.47). 후보 중 가장 높고 price_sum 1.182, distance 1.178, installments 1.100, seller_avg_delay 1.039 순. 테스트 1.279배 [1.220, 1.354]로 기저율과 구분됨.
- **2단계 모델**: 로지스틱 1.352배 [1.290, 1.436], R1 대비 +0.073 [0.026, 0.116]. RF 1.349배 [1.285, 1.435], R1 대비 +0.070 [0.017, 0.125]. D-21 규칙상 단일 변수보다 높다고 말할 수 있으나 개선폭은 약 0.07배(PR-AUC로는 약 0.007)이고, 기저율 위 초과분 0.352 중 `freight_sum` 단독이 0.279(약 79%)를 재현함.
- **정리**: 베이스라인(R0 기저율, R1 단일 변수) 대비 1단계는 로지스틱만 구분되고 개선폭도 작으며(PR-AUC 0.0523 vs 기저율 0.0440), RF는 구분되지 않음. 2단계는 두 모델 모두 기저율보다 높고 R1 대비 소폭 개선.
- **한계**: ① 주문 단위 독립 가정(1단계 테스트 양성의 약 75%가 5월·8월에 몰려 있어 구간이 낙관적) ② 모델 4개·변수 후보 여러 개를 동시에 비교해 다중비교 보정 없음 ③ 1단계 모델−R1 순위가 CV와 테스트에서 반대여서 기간에 따라 달라지는 결과임 ④ 어떤 변수 조합이 로지스틱의 개선을 만드는지는 분해하지 않음.


### 7-8 해석 (VIF, 모델 입력 변수만)

판정 기준은 코드에 명시(시점 미증명, VIF ≥ 5 주의, ≥ 10 심각). 학습 구간만 사용.

- 1단계 변수(n=70,852): 전부 1.00~1.01.
- 2단계 변수(n=70,319): `freight_sum` 1.35, `price_sum` 1.31, `distance_km_filled` 1.12, `installments_max` 1.12, 나머지 1.00~1.02.
- 카테고리 더미 포함(기준 범주 cat_bed_bath_table): 최대 `cat_기타` 1.80, `freight_sum` 1.45. 상위 10개 모두 2 미만.
- **결론**: 모델 입력 변수에서 다중공선성은 문제가 되지 않음. 04 7-1의 VIF가 입력 금지 변수(`delivery_error_days`, `expected_lead_days`)를 포함해 계산됐던 한계는 이 재계산으로 해소됨. 사전에 `freight_sum`과 `distance_km_filled`의 상관을 의심했으나 VIF 기준으로는 약했음(distance 1.12).
- 한계: VIF는 입력 변수 간 선형 상관만 보며 예측 성능이나 변수 중요도의 안정성을 보장하지 않음.


In [32]:
# 7-9. 민감도 분석 (설정 변경 없음, 테스트 구간 D-03도 그대로 — 결론을 바꾸는 용도가 아니라 보고용 숫자)
# (a) 테스트 월별·8월 제외 PR-AUC/기저율  (b) 학습에서 '도착일이 5/1 이후'인 주문을 뺀 재학습(분할 gap 민감도)
def _probs(tr, te, ycol, cols, splits, models):
    cv = single_var_cv(tr, ycol, cols, splits); var = cv.loc[0, "var"]
    s = _sign(tr[var].to_numpy(), tr[ycol].astype(int).to_numpy())
    probs = {f"R1 {var}({'+' if s > 0 else '-'})": s * te[var].to_numpy()}
    for name, mk, num, bin_, cat in models: probs[name] = fit_prob(mk, tr, ycol, num, bin_, cat, te)
    return probs

def month_sensitivity(label, probs, te, ycol):
    y = te[ycol].astype(int).to_numpy(); ym = te["purchase_ts"].dt.to_period("M").astype(str).to_numpy()
    groups = {"전체": np.ones(len(te), bool), "2018-05": ym == "2018-05", "2018-06": ym == "2018-06",
              "2018-07": ym == "2018-07", "2018-08": ym == "2018-08", "8월 제외": ym != "2018-08"}
    rows = []
    for g, m in groups.items():
        base = y[m].mean(); r = {"구간": g, "n": int(m.sum()), "양성": int(y[m].sum()), "기저율": base}
        for k, p in probs.items(): r[k] = average_precision_score(y[m], p[m]) / base
        rows.append(r)
    print(f"[{label}] 테스트 월별 PR-AUC/기저율 (월별은 양성이 적은 달이 불안정, 참고용)")
    print(pd.DataFrame(rows).round(3).to_string(index=False))

M1 = [(f"모델 {sel1_logit}", mk1l, num1, bin1, cat1), (f"모델 {sel1_rf}", mk1r, num1, bin1, cat1)]
M2 = [(f"모델 {sel2_logit}", mk2l, num2, bin1, cat1), (f"모델 {sel2_rf}", mk2r, num2, bin1, cat1)]
p1 = _probs(tr1, te1, "late", STAGE1_NUM, splits1, M1); month_sensitivity("1단계 late", p1, te1, "late"); print()
p2 = _probs(tr2, te2, "low", STAGE1_NUM + STAGE2_NUM, splits2, M2); month_sensitivity("2단계 low", p2, te2, "low")

# (b) gap 민감도: 학습 행 중 도착일이 테스트 시작(2018-05-01) 이후인 주문은 그 시점엔 라벨을 알 수 없었음
cut = pd.Timestamp("2018-05-01")
for label, tr, te, ycol, models in [("1단계 late", tr1, te1, "late", M1), ("2단계 low", tr2, te2, "low", M2)]:
    keep = tr["order_delivered_customer_date"] < cut
    y = te[ycol].astype(int).to_numpy(); base = y.mean()
    print(f"\n[{label}] 학습 {len(tr):,}건 중 도착일 ≥ 5/1: {(~keep).sum():,}건 ({(~keep).mean()*100:.2f}%), 그중 양성 {int(tr.loc[~keep, ycol].sum()):,}건 / 학습 양성 {int(tr[ycol].sum()):,}건")
    for name, mk, num, bin_, cat in models:
        p_all = fit_prob(mk, tr, ycol, num, bin_, cat, te); p_gap = fit_prob(mk, tr[keep], ycol, num, bin_, cat, te)
        print(f"  {name}: 전체 학습 {average_precision_score(y, p_all)/base:.3f} → 제외 후 {average_precision_score(y, p_gap)/base:.3f}")


[1단계 late] 테스트 월별 PR-AUC/기저율 (월별은 양성이 적은 달이 불안정, 참고용)
     구간     n   양성   기저율  R1 distance_km_filled(+)  모델 logit C=0.01  모델 rf depth=4 leaf=50
     전체 25352 1115 0.044                     1.054            1.189                  1.038
2018-05  6749  443 0.066                     1.439            1.511                  1.380
2018-06  6096   71 0.012                     2.004            1.467                  1.596
2018-07  6156  208 0.034                     1.367            1.868                  1.244
2018-08  6351  393 0.062                     0.665            0.774                  0.705
  8월 제외 19001  722 0.038                     1.405            1.567                  1.318

[2단계 low] 테스트 월별 PR-AUC/기저율 (월별은 양성이 적은 달이 불안정, 참고용)
     구간     n   양성   기저율  R1 freight_sum(+)  모델 logit C=0.01 cw=None  모델 rf depth=8 leaf=50 cw=None
     전체 25242 2525 0.100              1.279                    1.352                          1.349
2018-05  6722  723 0.108              1.338            

### 7-9 해석 (민감도 분석 — 테스트 결과를 본 뒤의 사후 분석, 설정과 D-03 테스트 구간은 변경 없음)

| PR-AUC/기저율 | 전체 | 8월 제외 | 8월 |
|---|---|---|---|
| 1단계 R1 distance | 1.054 | 1.405 | 0.665 |
| 1단계 로지스틱 | 1.189 | 1.567 | 0.774 |
| 1단계 RF | 1.038 | 1.318 | 0.705 |
| 2단계 R1 freight_sum | 1.279 | 1.295 | 1.240 |
| 2단계 로지스틱 | 1.352 | 1.399 | 1.189 |
| 2단계 RF | 1.349 | 1.365 | 1.329 |

- **1단계**: 5·6·7월은 세 방법 모두 월별 기저율 위(1.24~2.00배, 6월은 양성 71건이라 불안정)이고 8월만 기저율 아래(0.67~0.77배)다. 합산 성능이 낮은 것은 8월이 끌어내린 영향이 크다. 8월 제외 시 1.32~1.57배로 올라 선택 모델의 학습 CV(1.35~1.42배)에 가까워진다(모델 간 순위는 CV와 여전히 다르다). 이 결과는 6-10·6-11의 8월 진단(근거리 지연 급증, 예상일 단축)과 일관된다.
- **주의**: 8월 제외는 결과를 본 뒤의 사후 선택이라 주 결과는 전체 기준으로 두고 병기한다. 8월을 제외할 사전 근거는 없다. 월별 값은 월마다 기저율이 달라(1.2%~6.6%) 비교에 한계가 있다.
- **2단계**: 월별로 모델 1.19~1.51배로 일관되고 8월 제외 시에도 1.35 → 1.40배라 결론이 거의 안 바뀐다. 모델이 R1(`freight_sum`)보다 나은 정도는 월마다 다르고(5월 +0.18, 6월 +0.03, 7월 +0.07, 8월 로지스틱 −0.05) 일관되지 않다.
- **gap 민감도**: 학습 행의 약 3.95%(2,800건, 양성 472건)는 도착일이 5/1 이후라 그 시점엔 라벨을 알 수 없었다. 이 주문을 빼고 재학습해도 1단계 로지스틱 1.189 → 1.192, RF 1.038 → 1.022, 2단계 1.352 → 1.362, 1.349 → 1.351로 변화가 0.02 이내라 영향은 작다(2단계는 리뷰 응답 시점 때문에 근사).


### 7-10. 08용 산출물 저장 (사후 분석 입력, 설정·테스트 구간 변경 없음)

In [33]:
# 7-10. 08용 산출물 저장
import json, os
OUT = "data/processed/07_outputs"; os.makedirs(OUT, exist_ok=True)

# (stage, model, 선택된 설정명, make_clf, 학습, 테스트, 라벨, num) — cv_results 순서와 동일
SPECS = [
    ("stage1", "logit", sel1_logit, mk1l, tr1, te1, "late", num1),
    ("stage1", "rf",    sel1_rf,    mk1r, tr1, te1, "late", num1),
    ("stage2", "logit", sel2_logit, mk2l, tr2, te2, "low",  num2),
    ("stage2", "rf",    sel2_rf,    mk2r, tr2, te2, "low",  num2),
]
assert len(cv_results) == 4

meta = {"train_end": TRAIN_END, "models": {}}
frames = {}
for (stage, mdl, sel, mk, tr, te, ycol, num), cvr in zip(SPECS, cv_results):
    p = fit_prob(mk, tr, ycol, num, bin1, cat1, te)
    y = te[ycol].astype(int).to_numpy()
    pr = average_precision_score(y, p)
    assert abs(pr - cvr["pr_auc"]) < 1e-6, f"{stage} {mdl}: 재적합 PR-AUC {pr:.6f} != 7절 {cvr['pr_auc']:.6f}"
    if stage not in frames:
        frames[stage] = pd.DataFrame({"order_id": te["order_id"].to_numpy(), "y": y})
    frames[stage][f"prob_{mdl}"] = p
    meta["models"][f"{stage}_{mdl}"] = {"selected": sel, "thr": float(cvr["thr"]),
                                        "pr_auc": float(pr), "base_rate": float(y.mean()), "n_test": int(len(y))}
for stage, df in frames.items():
    df.to_parquet(f"{OUT}/preds_{stage}.parquet", index=False)
pd.DataFrame(cv_results).to_csv(f"{OUT}/cv_results.csv", index=False)
json.dump(meta, open(f"{OUT}/meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("저장 완료:", sorted(os.listdir(OUT)))


저장 완료: ['cv_results.csv', 'meta.json', 'preds_stage1.parquet', 'preds_stage2.parquet']
